# DefectSpotter — good-only manufacturing visual inspection

This notebook builds a complete anomaly-inspection project for **bottle**, **screw**, and **carpet** images from MVTec AD. It trains only on good parts, compares a PatchCore-style feature-memory detector with a convolutional VAE and an aligned mean-image baseline, localizes suspected defects, calibrates a cost-sensitive PASS/CHECK/REJECT policy, evaluates drift, exports ONNX, packages a local project ZIP, stores checkpoints in Google Drive, and launches a Gradio demonstration.

## What this rebuilt notebook produces

- EDA tables, montages, acquisition statistics, defect-mask analysis, and synthetic-defect galleries.
- Good-only model training plus bounded hyperparameter search.
- Procedural cut-paste, scratch, and stain anomalies, with optional diffusion inpainting.
- Image AUROC, average precision, pixel AUROC, AUPRO@0.30, confidence intervals, false-reject rate, escape rate, review load, expected cost, and latency.
- Synthetic-only threshold calibration compared with a labelled-real-defect oracle threshold.
- A verified MobileNetV3-Small ONNX feature extractor and compact per-category memory banks.
- A local project ZIP with source, requirements, tests, docs, reports, figures, and trained artifacts.
- A Gradio UI that opens with EDA/results already loaded and exposes `/inspect_part` and `/inspect_batch`.

> **No benchmark values are prefilled.** All evidence is generated by the notebook run.

## Recommended execution

1. Open in Google Colab and select a GPU runtime when available.
2. Leave `PROFILE = "smoke"` for the first complete **Runtime → Run all** validation.
3. After the smoke run succeeds, change to `PROFILE = "balanced"` for the research run.
4. Keep the Colab runtime connected while using the temporary Gradio public URL.


## License, deployment, and safety boundary

MVTec AD is licensed **CC BY-NC-SA 4.0** and the official dataset page states that commercial use is not allowed. This notebook is therefore a research and teaching implementation. A real factory deployment must use plant-owned or appropriately licensed data, qualify the camera and illumination, validate each critical defect mode, secure production images, log operator overrides, and assign ownership of the CHECK/REJECT process to a quality engineer.

A heat map identifies visual deviation; it does not prove root cause. `CHECK` is intentionally human-reviewed. A drift alarm triggers imaging-system investigation and controlled revalidation—not silent model updating.

## 1 · Install and verify dependencies

Colab already provides a CUDA-compatible PyTorch build. This cell installs the portable inference, UI, data-download, and evaluation dependencies without replacing PyTorch. It then verifies the core modules before any expensive work begins.

In [ ]:
# Install packages that are not reliably preinstalled in Colab.
# PyTorch and torchvision are intentionally kept from the active Colab runtime
# to avoid replacing a compatible CUDA build with a mismatched wheel.
from __future__ import annotations

import importlib
import importlib.util
import json
import os
import subprocess
import sys

os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

required_packages = [
    "huggingface-hub>=0.27,<2",
    "onnx>=1.16,<2",
    "onnxscript>=0.1,<1",
    "onnxruntime>=1.19,<2",
    "faiss-cpu>=1.8,<2",
    "gradio>=5,<7",
    "streamlit>=1.40,<2",
    "plotly>=5.24,<7",
    "PyYAML>=6,<7",
    "tqdm>=4.66,<5",
    "scikit-image>=0.22,<1",
    "pytest>=8,<10",
    "packaging>=24,<27",
]

print("Installing and validating runtime dependencies...")
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "--upgrade-strategy",
        "only-if-needed",
        *required_packages,
    ]
)
importlib.invalidate_caches()

module_names = [
    "torch", "torchvision", "cv2", "numpy", "pandas", "sklearn", "skimage",
    "gradio", "onnx", "onnxruntime", "faiss", "huggingface_hub", "streamlit",
]
missing = [name for name in module_names if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Required modules were not found after installation: {missing}")

versions = {}
for name in module_names:
    module = importlib.import_module(name)
    versions[name] = getattr(module, "__version__", "installed")
print(json.dumps(versions, indent=2, default=str) if "json" in globals() else versions)
print("Dependency check completed.")


## 2 · Configure the experiment

This is the main control panel. Runtime profiles bound compute while keeping the same end-to-end workflow:

- **smoke:** reduced data and epochs for integration testing;
- **balanced:** default research run across all selected images;
- **thorough:** larger hyperparameter grids, more VAE training, and stronger uncertainty estimates.

Cost assumptions are explicit because the economically correct threshold depends on defect prevalence, escape consequences, scrap/rework cost, and inspector capacity.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import os

# ---------------------------- USER CONTROLS ----------------------------
PROFILE = "smoke"          # "smoke", "balanced", or "thorough"
CATEGORIES = ["bottle", "screw", "carpet"]
SEED = 42
RESUME_RUN_ID = None           # e.g. "20261117_120000_balanced" to resume a Drive run
MANUAL_ARCHIVE_IN_DRIVE = ""  # optional .zip/.tar path if the mirror cannot be used
ENABLE_DIFFUSION = False       # optional and GPU-heavy; procedural defects remain the default
DIFFUSION_SAMPLES_PER_CATEGORY = 4
FORCE_RETRAIN = False          # True ignores reusable per-category artifacts
FORCE_REBUILD_ONNX = False     # True forces a fresh ONNX export
AUTO_DOWNLOAD_ZIP = True       # also stores the ZIP in Drive
AUTO_LAUNCH_GRADIO = True      # final cell launches the UI automatically
GRADIO_SHARE = True            # temporary public gradio.live tunnel

# Hugging Face deployment. The token is read securely from the Colab secret HF_TOKEN
# or requested with a hidden prompt; it is never written into this notebook.
HF_USERNAME = "bhkr0901"
HF_SPACE_NAME = "defectspotter"
HF_MODEL_REPO_NAME = "defectspotter-artifacts"
HF_REPOSITORIES_PRIVATE = False
HF_SPACE_HARDWARE = "zero-a10g"  # eligible free accounts; fallback uses the account default
DEPLOY_TO_HUGGINGFACE = True
HF_WAIT_FOR_BUILD = True
HF_BUILD_TIMEOUT_SECONDS = 900

# Cost assumptions are deliberately editable. They are examples, not universal truths.
DEFECT_PREVALENCE = 0.01
COST_ESCAPE = 200.0
COST_FALSE_REJECT = 1.0
COST_HUMAN_REVIEW = 0.15
CHECK_BAND = 5.0

HF_REPO_ID = "foersben/mvtec-ad"
HF_PINNED_REVISION = "14f50140550afeccd97ad31d4cf68fbf9bb94e37"

PROFILE_CONFIGS = {
    "smoke": {
        "patch_image_size": 160,
        "feature_batch_size": 16,
        "train_cap": 60,
        "test_cap": 60,
        "cv_cap": 40,
        "cv_folds": 2,
        "memory_ratios": [0.05],
        "topk_fractions": [0.01],
        "blur_sigmas": [2.5],
        "max_memory_mb": 20.0,
        "synthetic_calibration_cap": 30,
        "vae_image_size": 96,
        "vae_candidates": [
            {"latent_dim": 32, "base_channels": 16, "beta": 1e-4, "learning_rate": 2e-4},
        ],
        "vae_search_epochs": 2,
        "vae_final_epochs": 4,
        "vae_batch_size": 32,
        "bootstrap_repeats": 100,
        "pixel_max_samples": 250_000,
        "aupro_thresholds": 35,
        "latency_repeats": 5,
        "drift_samples": 10,
    },
    "balanced": {
        "patch_image_size": 224,
        "feature_batch_size": 16,
        "train_cap": None,
        "test_cap": None,
        "cv_cap": 80,
        "cv_folds": 2,
        "memory_ratios": [0.03, 0.06],
        "topk_fractions": [0.005, 0.01],
        "blur_sigmas": [2.0, 4.0],
        "max_memory_mb": 45.0,
        "synthetic_calibration_cap": 80,
        "vae_image_size": 128,
        "vae_candidates": [
            {"latent_dim": 32, "base_channels": 24, "beta": 1e-4, "learning_rate": 2e-4},
            {"latent_dim": 64, "base_channels": 24, "beta": 1e-4, "learning_rate": 2e-4},
            {"latent_dim": 64, "base_channels": 32, "beta": 5e-4, "learning_rate": 1e-4},
        ],
        "vae_search_epochs": 3,
        "vae_final_epochs": 15,
        "vae_batch_size": 32,
        "bootstrap_repeats": 500,
        "pixel_max_samples": 1_000_000,
        "aupro_thresholds": 80,
        "latency_repeats": 15,
        "drift_samples": 24,
    },
    "thorough": {
        "patch_image_size": 256,
        "feature_batch_size": 16,
        "train_cap": None,
        "test_cap": None,
        "cv_cap": 140,
        "cv_folds": 3,
        "memory_ratios": [0.02, 0.05, 0.10],
        "topk_fractions": [0.005, 0.01, 0.02],
        "blur_sigmas": [2.0, 4.0, 6.0],
        "max_memory_mb": 48.0,
        "synthetic_calibration_cap": 160,
        "vae_image_size": 128,
        "vae_candidates": [
            {"latent_dim": 32, "base_channels": 24, "beta": 1e-4, "learning_rate": 2e-4},
            {"latent_dim": 64, "base_channels": 24, "beta": 1e-4, "learning_rate": 2e-4},
            {"latent_dim": 64, "base_channels": 32, "beta": 5e-4, "learning_rate": 1e-4},
            {"latent_dim": 96, "base_channels": 32, "beta": 1e-4, "learning_rate": 1e-4},
            {"latent_dim": 128, "base_channels": 32, "beta": 1e-3, "learning_rate": 1e-4},
            {"latent_dim": 64, "base_channels": 40, "beta": 1e-4, "learning_rate": 2e-4},
        ],
        "vae_search_epochs": 5,
        "vae_final_epochs": 30,
        "vae_batch_size": 32,
        "bootstrap_repeats": 1500,
        "pixel_max_samples": 2_000_000,
        "aupro_thresholds": 120,
        "latency_repeats": 30,
        "drift_samples": 45,
    },
}

if PROFILE not in PROFILE_CONFIGS:
    raise ValueError(f"Unknown PROFILE={PROFILE!r}; choose from {sorted(PROFILE_CONFIGS)}")
CFG = PROFILE_CONFIGS[PROFILE]

RUN_ID = RESUME_RUN_ID or datetime.now(timezone.utc).strftime(f"%Y%m%d_%H%M%S_{PROFILE}")
print(json.dumps({"run_id": RUN_ID, "profile": PROFILE, "categories": CATEGORIES, "hf_space": f"{HF_USERNAME}/{HF_SPACE_NAME}", "hf_model_repo": f"{HF_USERNAME}/{HF_MODEL_REPO_NAME}", **CFG}, indent=2))


## 3 · Mount Drive and create resumable run folders

The notebook writes models, manifests, figures, reports, and stage status to a unique Drive run directory. File updates use temporary files and atomic replacement where practical. Set `RESUME_RUN_ID` in the previous cell to restore completed per-category artifacts and caches from an earlier run.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
from pathlib import Path

IN_COLAB = False
DRIVE_MOUNTED = False
colab_files = None
try:
    from google.colab import drive as colab_drive
    from google.colab import files as colab_files
    IN_COLAB = True
    try:
        colab_drive.mount("/content/drive", force_remount=False)
        DRIVE_MOUNTED = Path("/content/drive/MyDrive").exists()
    except Exception as exc:
        print(f"Google Drive could not be mounted; continuing with session storage: {exc}")
except Exception:
    pass

SESSION_ROOT = Path("/content/defectspotter_workspace") if IN_COLAB else Path.cwd() / "defectspotter_workspace"
DRIVE_BASE = (
    Path("/content/drive/MyDrive/DefectSpotter")
    if DRIVE_MOUNTED
    else SESSION_ROOT / "Drive_Fallback" / "DefectSpotter"
)
DRIVE_RUN_DIR = DRIVE_BASE / "runs" / RUN_ID
PROJECT_DIR = SESSION_ROOT / "DefectSpotter_Project"
DATA_DIR = SESSION_ROOT / "data" / "mvtec_ad"
WORK_CACHE_DIR = SESSION_ROOT / "work_cache"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
REPORT_DIR = PROJECT_DIR / "reports"
FIGURE_DIR = PROJECT_DIR / "figures"
MANIFEST_DIR = PROJECT_DIR / "manifests"

for folder in [
    SESSION_ROOT, DRIVE_RUN_DIR, PROJECT_DIR, WORK_CACHE_DIR, ARTIFACT_DIR,
    REPORT_DIR, FIGURE_DIR, MANIFEST_DIR,
    DRIVE_RUN_DIR / "checkpoints", DRIVE_RUN_DIR / "artifacts",
    DRIVE_RUN_DIR / "figures", DRIVE_RUN_DIR / "reports", DRIVE_RUN_DIR / "logs",
]:
    folder.mkdir(parents=True, exist_ok=True)

STATUS_PATH = DRIVE_RUN_DIR / "stage_status.json"


def atomic_write_json(payload, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str), encoding="utf-8")
    os.replace(temporary, path)


def load_stage_status() -> dict:
    if STATUS_PATH.exists():
        return json.loads(STATUS_PATH.read_text(encoding="utf-8"))
    return {"run_id": RUN_ID, "profile": PROFILE, "stages": {}}


def mark_stage(stage: str, state: str = "complete", **details) -> None:
    status = load_stage_status()
    status.setdefault("stages", {})[stage] = {"state": state, **details}
    atomic_write_json(status, STATUS_PATH)
    print(f"[checkpoint] {stage}: {state} -> {STATUS_PATH}")


def copy_checkpoint(source: str | Path, relative_destination: str | Path) -> Path:
    source = Path(source)
    if not source.exists():
        raise FileNotFoundError(f"Checkpoint source does not exist: {source}")
    destination = DRIVE_RUN_DIR / relative_destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source, destination)
    else:
        shutil.copy2(source, destination)
    print(f"[drive] {source} -> {destination}")
    return destination


def restore_checkpoint(relative_source: str | Path, destination: str | Path) -> bool:
    source = DRIVE_RUN_DIR / relative_source
    destination = Path(destination)
    if not source.exists():
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source, destination)
    else:
        shutil.copy2(source, destination)
    print(f"[restore] {source} -> {destination}")
    return True

RUN_CONFIG = {
    "run_id": RUN_ID,
    "profile": PROFILE,
    "categories": CATEGORIES,
    "seed": SEED,
    "defect_prevalence": DEFECT_PREVALENCE,
    "cost_escape": COST_ESCAPE,
    "cost_false_reject": COST_FALSE_REJECT,
    "cost_human_review": COST_HUMAN_REVIEW,
    "check_band": CHECK_BAND,
    "enable_diffusion": ENABLE_DIFFUSION,
    "force_retrain": FORCE_RETRAIN,
    "force_rebuild_onnx": FORCE_REBUILD_ONNX,
    "profile_config": CFG,
    "hf_repo_id": HF_REPO_ID,
    "hf_revision": HF_PINNED_REVISION,
    "hf_username": HF_USERNAME,
    "hf_space_repo": f"{HF_USERNAME}/{HF_SPACE_NAME}",
    "hf_model_repo": f"{HF_USERNAME}/{HF_MODEL_REPO_NAME}",
    "hf_space_hardware": HF_SPACE_HARDWARE,
    "drive_mounted": DRIVE_MOUNTED,
}
atomic_write_json(RUN_CONFIG, DRIVE_RUN_DIR / "config.json")
atomic_write_json(RUN_CONFIG, PROJECT_DIR / "run_config.json")
mark_stage("drive_and_paths", session_root=str(SESSION_ROOT), drive_run_dir=str(DRIVE_RUN_DIR), drive_mounted=DRIVE_MOUNTED)

print(f"Session workspace: {SESSION_ROOT}")
print(f"Persistent run folder: {DRIVE_RUN_DIR}")
print(f"Local project folder: {PROJECT_DIR}")
print(f"Google Drive mounted: {DRIVE_MOUNTED}")


## 4 · Record the execution environment

Reproducibility requires more than a seed. This cell records Python, PyTorch, torchvision, CUDA, ONNX Runtime, available providers, and a full `pip freeze` in both the session project and Drive.

In [ ]:
from __future__ import annotations

import os
import platform
import subprocess
import sys
from pathlib import Path

import torch
import torchvision

try:
    import onnxruntime as ort
    ort_version = ort.__version__
    ort_providers = ort.get_available_providers()
except Exception as exc:
    ort_version = f"unavailable: {exc}"
    ort_providers = []

ENVIRONMENT = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cuda_runtime": torch.version.cuda,
    "onnxruntime": ort_version,
    "onnxruntime_providers": ort_providers,
}

atomic_write_json(ENVIRONMENT, DRIVE_RUN_DIR / "environment.json")
atomic_write_json(ENVIRONMENT, PROJECT_DIR / "environment.json")

try:
    freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
except Exception as exc:
    freeze = f"Could not capture pip freeze: {exc}"
(DRIVE_RUN_DIR / "environment_freeze.txt").write_text(freeze, encoding="utf-8")
(PROJECT_DIR / "environment_freeze.txt").write_text(freeze, encoding="utf-8")

print(ENVIRONMENT)
mark_stage("environment_recorded", **ENVIRONMENT)


## 5 · Materialize the self-contained project source

The notebook contains the complete Python package, Gradio app, Streamlit app, tests, requirements, configuration, documentation, and research report. This cell writes those files into the Colab session. The final ZIP therefore remains locally runnable without relying on hidden notebook definitions.

In [ ]:
from __future__ import annotations

import base64
import hashlib
import io
import zipfile
from pathlib import Path

SOURCE_ARCHIVE_SHA256 = "36a66e03be355aace72e36d6ef6f83af6564a6c12d97fd44f0ad74a707b3e545"
SOURCE_ARCHIVE_FILE_COUNT = 30
SOURCE_ARCHIVE_B64 = """..."""

archive_bytes = base64.b64decode(SOURCE_ARCHIVE_B64.encode("ascii"))
actual_sha = hashlib.sha256(archive_bytes).hexdigest()
if actual_sha != SOURCE_ARCHIVE_SHA256:
    raise RuntimeError(f"Embedded source archive checksum mismatch: {actual_sha} != {SOURCE_ARCHIVE_SHA256}")

project_root_resolved = PROJECT_DIR.resolve()
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = [member for member in archive.infolist() if not member.is_dir()]
    for member in members:
        destination = (PROJECT_DIR / member.filename).resolve()
        if not destination.is_relative_to(project_root_resolved):
            raise RuntimeError(f"Unsafe archive member: {member.filename}")
    archive.extractall(PROJECT_DIR)

written_files = sorted(path.relative_to(PROJECT_DIR).as_posix() for path in PROJECT_DIR.rglob("*") if path.is_file())
source_files = [name for name in written_files if not name.startswith(("artifacts/", "figures/", "manifests/"))]
print(f"Extracted {len(members)} verified source files to {PROJECT_DIR}")
if len(members) != SOURCE_ARCHIVE_FILE_COUNT:
    raise RuntimeError(f"Expected {SOURCE_ARCHIVE_FILE_COUNT} source files but extracted {len(members)}")
for name in source_files:
    print(f" - {name}")
mark_stage("source_package_written", files=len(members), archive_sha256=actual_sha)


## 6 · Import the project and set deterministic controls

Core modules are imported from the package that was just written. The notebook selects GPU when available and sets Python, NumPy, and PyTorch seeds. Exact bitwise equality across every GPU/library version is not promised; the environment record makes differences auditable.

In [ ]:
from __future__ import annotations

import gc
import importlib
import itertools
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import torch
from sklearn.model_selection import KFold, train_test_split
from torch.utils.data import DataLoader

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
importlib.invalidate_caches()

from defectspotter.data import (
    RGBImageDataset,
    download_mvtec_categories,
    extract_archive,
    generate_synthetic_records,
    image_statistics,
    index_mvtec,
    make_synthetic_defect,
    read_mask,
    read_rgb,
    resize_rgb,
    save_json,
    set_global_seed,
    validate_mvtec_layout,
)
from defectspotter.evaluation import (
    benchmark_latency,
    bootstrap_auc_ci,
    choose_cost_threshold,
    compute_aupro,
    drift_alert_table,
    image_metrics,
    sampled_pixel_auroc,
    safe_roc_auc,
    three_way_operating_metrics,
    threshold_distance,
)
from defectspotter.models import (
    ConvVAE,
    MeanBaselineConfig,
    MeanImageBaseline,
    MobileNetPatchExtractor,
    VAEConfig,
    build_memory_bank,
    export_feature_extractor_onnx,
    extract_patch_maps,
    load_vae,
    normalize_with_scaler,
    robust_scaler,
    score_patch_maps,
    train_vae,
    vae_predict,
)

set_global_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Compute device: {DEVICE}")
if DEVICE == "cpu" and PROFILE != "smoke":
    print("WARNING: balanced/thorough execution on CPU can take a long time. Use PROFILE=\"smoke\" for the first full run.")
print(f"Project package imported from: {PROJECT_DIR}")


### 6A · Run package compatibility tests before expensive training

These tests exercise synthetic-defect generation, NumPy 1.x/2.x AUPRO compatibility, cost-aware decisions, VAE checkpoint resume, packaged inference, and Gradio construction. A failure here stops before data download or model training.

In [ ]:
from __future__ import annotations

import subprocess

compatibility_test = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_DIR,
    text=True,
    capture_output=True,
)
print(compatibility_test.stdout)
if compatibility_test.returncode != 0:
    print(compatibility_test.stderr)
    raise RuntimeError("Source-package compatibility tests failed. Training was not started.")
mark_stage("source_compatibility_tests", tests="passed")


## 7 · Download and validate MVTec AD

The official MVTec page is authoritative but its download flow may not expose a stable unattended URL. For reproducibility, the notebook downloads only the three selected folders from a non-commercial Hugging Face mirror, first trying a pinned revision and then `main`. A manual Drive archive path is available as a fallback.

Validation checks training-good images, test defects, defect groups, and pixel-mask availability before continuing.

In [ ]:
from __future__ import annotations

import traceback


def locate_mvtec_root(base: Path) -> Path | None:
    candidates = [base]
    candidates.extend(p for p in base.glob("*") if p.is_dir())
    candidates.extend(p for p in base.glob("*/*") if p.is_dir())
    for candidate in candidates:
        if all((candidate / category / "train" / "good").exists() for category in CATEGORIES):
            return candidate
    return None


def ensure_dataset() -> Path:
    existing = locate_mvtec_root(DATA_DIR)
    if existing is not None:
        print(f"Using existing MVTec AD folders at {existing}")
        return existing

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    download_errors = []
    for revision in [HF_PINNED_REVISION, None]:
        try:
            print(f"Downloading selected categories from {HF_REPO_ID} at revision={revision or 'main'}")
            download_mvtec_categories(DATA_DIR, CATEGORIES, repo_id=HF_REPO_ID, revision=revision)
            found = locate_mvtec_root(DATA_DIR)
            if found is not None:
                return found
        except Exception as exc:
            download_errors.append(f"revision={revision}: {exc}")
            print(traceback.format_exc(limit=2))

    if MANUAL_ARCHIVE_IN_DRIVE:
        archive = Path(MANUAL_ARCHIVE_IN_DRIVE)
        if archive.exists():
            print(f"Mirror download failed; extracting manual archive {archive}")
            extract_archive(archive, DATA_DIR)
            found = locate_mvtec_root(DATA_DIR)
            if found is not None:
                return found
        else:
            download_errors.append(f"Manual archive does not exist: {archive}")

    raise RuntimeError(
        "Could not obtain the selected MVTec AD categories. Errors:\n- " + "\n- ".join(download_errors)
    )


MVTEC_ROOT = ensure_dataset()
layout_summary = validate_mvtec_layout(MVTEC_ROOT, CATEGORIES)
index_df = index_mvtec(MVTEC_ROOT, CATEGORIES)

summary_rows = []
for category, values in layout_summary.items():
    summary_rows.append({"category": category, **values, "defect_types": ", ".join(values["defect_types"])})
dataset_summary = pd.DataFrame(summary_rows)

dataset_summary.to_csv(MANIFEST_DIR / "dataset_summary.csv", index=False)
index_df.to_csv(MANIFEST_DIR / "dataset_index.csv", index=False)
copy_checkpoint(MANIFEST_DIR / "dataset_summary.csv", "manifests/dataset_summary.csv")
copy_checkpoint(MANIFEST_DIR / "dataset_index.csv", "manifests/dataset_index.csv")

print(dataset_summary.to_string(index=False))
print(f"Indexed {len(index_df):,} images from {MVTEC_ROOT}")
print("License reminder: MVTec AD is CC BY-NC-SA 4.0 and is not permitted for commercial use.")
mark_stage("dataset_validated", mvtec_root=str(MVTEC_ROOT), rows=len(index_df), summary=layout_summary)


## 8 · Create leakage-resistant splits

Good training images are divided into a model-training pool and an untouched good-calibration pool. The official test images are separated into:

- an **oracle threshold calibration** subset used only to quantify how a labelled threshold differs from the synthetic-only threshold;
- an **operating test** subset used for false-reject, escape, review, and cost comparisons.

No real defect image trains a detector or selects its deployable threshold.

In [ ]:
from __future__ import annotations


def deterministic_group_cap(frame: pd.DataFrame, cap: int | None, group_column: str) -> pd.DataFrame:
    if cap is None or len(frame) <= cap:
        return frame.copy().reset_index(drop=True)
    groups = []
    group_sizes = frame[group_column].value_counts()
    allocations = {name: max(1, int(round(cap * count / len(frame)))) for name, count in group_sizes.items()}
    # Correct rounding while preserving at least one sample per group.
    while sum(allocations.values()) > cap:
        largest = max((k for k in allocations if allocations[k] > 1), key=lambda k: allocations[k], default=None)
        if largest is None:
            break
        allocations[largest] -= 1
    while sum(allocations.values()) < cap:
        largest = max(group_sizes.index, key=lambda k: group_sizes[k] - allocations.get(k, 0))
        allocations[largest] = allocations.get(largest, 0) + 1
    for group_name, group in frame.groupby(group_column):
        groups.append(group.sample(n=min(len(group), allocations[group_name]), random_state=SEED))
    return pd.concat(groups, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)


split_rows: list[dict] = []
for category in CATEGORIES:
    train_good = index_df[(index_df.category == category) & (index_df.split == "train")].copy()
    if CFG["train_cap"] is not None and len(train_good) > CFG["train_cap"]:
        train_good = train_good.sample(n=CFG["train_cap"], random_state=SEED).reset_index(drop=True)

    model_train, calibration_good = train_test_split(
        train_good,
        test_size=0.20,
        random_state=SEED,
        shuffle=True,
    )
    for role, frame in [("model_train_good", model_train), ("calibration_good", calibration_good)]:
        for row in frame.to_dict("records"):
            split_rows.append({**row, "role": role})

    test = index_df[(index_df.category == category) & (index_df.split == "test")].copy()
    test = deterministic_group_cap(test, CFG["test_cap"], "defect_type")
    # The oracle split is used only to compare a real-label threshold against the deployable synthetic-only threshold.
    # Stratify by defect type when every group has enough images so operating metrics retain each failure mode.
    defect_type_counts = test["defect_type"].value_counts()
    stratify_values = test["defect_type"] if int(defect_type_counts.min()) >= 2 else test["label"]
    oracle_cal, operating_test = train_test_split(
        test,
        test_size=0.65,
        random_state=SEED,
        stratify=stratify_values,
    )
    for role, frame in [("oracle_threshold_calibration", oracle_cal), ("operating_test", operating_test)]:
        for row in frame.to_dict("records"):
            split_rows.append({**row, "role": role})

split_manifest = pd.DataFrame(split_rows).sort_values(["category", "role", "defect_type", "image_path"]).reset_index(drop=True)
split_manifest.to_csv(MANIFEST_DIR / "splits.csv", index=False)
copy_checkpoint(MANIFEST_DIR / "splits.csv", "manifests/splits.csv")

print(split_manifest.groupby(["category", "role", "label"]).size().rename("count").reset_index().to_string(index=False))
mark_stage("splits_created", rows=len(split_manifest), seed=SEED)


## 9 · Exploratory data analysis

This cell measures resolution, brightness, contrast, sharpness, color statistics, and defect-mask area. It saves composition plots, acquisition-statistic plots, mask-size distributions, and category montages to both the project and Drive. These checks expose camera/domain differences that can later appear as drift.

In [ ]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed


def save_and_show_figure(figure: plt.Figure, name: str) -> Path:
    path = FIGURE_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path, dpi=160, bbox_inches="tight")
    shutil.copy2(path, DRIVE_RUN_DIR / "figures" / path.name)
    plt.show()
    plt.close(figure)
    print(f"Saved figure: {path}")
    return path


def compute_stats_row(record: dict) -> dict:
    stats = image_statistics(record["image_path"])
    mask_area = 0.0
    mask_path = record.get("mask_path")
    if isinstance(mask_path, str) and mask_path.strip():
        mask = read_mask(mask_path)
        mask_area = float(100.0 * mask.mean())
    return {**record, **stats, "mask_area_pct": mask_area}


records = index_df.to_dict("records")
statistics_rows = []
with ThreadPoolExecutor(max_workers=min(12, os.cpu_count() or 4)) as pool:
    futures = [pool.submit(compute_stats_row, record) for record in records]
    for index, future in enumerate(as_completed(futures), start=1):
        statistics_rows.append(future.result())
        if index % 200 == 0 or index == len(futures):
            print(f"Computed image statistics for {index}/{len(futures)} images")

image_stats_df = pd.DataFrame(statistics_rows)
image_stats_df.to_csv(REPORT_DIR / "image_statistics.csv", index=False)
copy_checkpoint(REPORT_DIR / "image_statistics.csv", "reports/image_statistics.csv")

# Dataset composition
composition = (
    index_df.assign(status=np.where(index_df.label.eq(1), "defect", "good"))
    .groupby(["category", "split", "status"])
    .size()
    .rename("images")
    .reset_index()
)
composition.to_csv(REPORT_DIR / "dataset_composition.csv", index=False)

fig, axis = plt.subplots(figsize=(9, 4.5))
pivot = composition.pivot_table(index="category", columns=["split", "status"], values="images", fill_value=0)
pivot.plot(kind="bar", ax=axis)
axis.set_title("MVTec AD subset composition")
axis.set_ylabel("Images")
axis.set_xlabel("Category")
axis.tick_params(axis="x", rotation=0)
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_and_show_figure(fig, "eda_dataset_composition.png")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for axis, metric in zip(axes, ["brightness", "contrast", "sharpness"]):
    image_stats_df.boxplot(column=metric, by="category", ax=axis, grid=False)
    axis.set_title(metric.title())
    axis.set_xlabel("")
fig.suptitle("Acquisition statistics by category")
fig.tight_layout()
save_and_show_figure(fig, "eda_acquisition_statistics.png")

anomalous_stats = image_stats_df[image_stats_df.label.eq(1)]
fig, axis = plt.subplots(figsize=(8, 4.5))
for category, group in anomalous_stats.groupby("category"):
    axis.hist(group["mask_area_pct"], bins=18, alpha=0.45, label=category)
axis.set_title("Ground-truth defect area as a percentage of the image")
axis.set_xlabel("Mask area (%)")
axis.set_ylabel("Images")
axis.legend()
axis.grid(alpha=0.2)
fig.tight_layout()
save_and_show_figure(fig, "eda_defect_mask_area.png")


def sample_montage(category: str, samples_per_status: int = 3) -> plt.Figure:
    good = index_df[(index_df.category == category) & (index_df.split == "train")].sample(
        n=min(samples_per_status, int(((index_df.category == category) & (index_df.split == "train")).sum())),
        random_state=SEED,
    )
    defects = index_df[(index_df.category == category) & index_df.label.eq(1)].sample(
        n=min(samples_per_status, int(((index_df.category == category) & index_df.label.eq(1)).sum())),
        random_state=SEED,
    )
    rows = max(len(good), len(defects))
    fig, axes = plt.subplots(rows, 3, figsize=(11, 3.3 * rows), squeeze=False)
    for row_index in range(rows):
        if row_index < len(good):
            good_row = good.iloc[row_index]
            axes[row_index, 0].imshow(read_rgb(good_row.image_path))
            axes[row_index, 0].set_title("Good training image")
        else:
            axes[row_index, 0].axis("off")
        if row_index < len(defects):
            defect_row = defects.iloc[row_index]
            axes[row_index, 1].imshow(read_rgb(defect_row.image_path))
            axes[row_index, 1].set_title(f"Defect: {defect_row.defect_type}")
            axes[row_index, 2].imshow(read_mask(defect_row.mask_path), cmap="gray")
            axes[row_index, 2].set_title("Pixel ground truth")
        else:
            axes[row_index, 1].axis("off")
            axes[row_index, 2].axis("off")
        for axis in axes[row_index]:
            axis.axis("off")
    fig.suptitle(f"{category.title()} examples")
    fig.tight_layout()
    return fig


for category in CATEGORIES:
    save_and_show_figure(sample_montage(category), f"eda_montage_{category}.png")

mark_stage("eda_complete", statistics_rows=len(image_stats_df), figures=len(list(FIGURE_DIR.glob("eda_*.png"))))


## 10 · Generate synthetic calibration anomalies

Held-out good images are transformed into cut-paste, scratch, and stain examples. Object anomalies are biased toward an estimated part foreground, while carpet uses the whole texture field. Every generated anomaly has a known mask and seed.

These images are used to select hyperparameters and the deployable threshold; they are **not** treated as proof of real-defect performance.

In [ ]:
from __future__ import annotations

SYNTHETIC_ROOT = SESSION_ROOT / "synthetic_calibration"
synthetic_frames = []

for category in CATEGORIES:
    category_good = split_manifest[
        (split_manifest.category == category) & (split_manifest.role == "calibration_good")
    ].copy()
    if len(category_good) > CFG["synthetic_calibration_cap"]:
        category_good = category_good.sample(n=CFG["synthetic_calibration_cap"], random_state=SEED)
    output_dir = SYNTHETIC_ROOT / category
    manifest_path = output_dir / "manifest.csv"
    if manifest_path.exists():
        frame = pd.read_csv(manifest_path)
        print(f"Reusing {len(frame)} synthetic {category} images from {manifest_path}")
    else:
        frame = generate_synthetic_records(
            category_good.image_path.tolist(),
            category=category,
            output_dir=output_dir,
            per_image=1,
            seed=SEED,
        )
    synthetic_frames.append(frame)
    copy_checkpoint(output_dir, Path("checkpoints") / "synthetic" / category)

SYNTHETIC_MANIFEST = pd.concat(synthetic_frames, ignore_index=True)
SYNTHETIC_MANIFEST.to_csv(MANIFEST_DIR / "synthetic_manifest.csv", index=False)
copy_checkpoint(MANIFEST_DIR / "synthetic_manifest.csv", "manifests/synthetic_manifest.csv")

print(SYNTHETIC_MANIFEST.groupby(["category", "synthetic_type"]).size().rename("images").reset_index().to_string(index=False))

fig, axes = plt.subplots(len(CATEGORIES), 4, figsize=(13, 3.4 * len(CATEGORIES)), squeeze=False)
for row_index, category in enumerate(CATEGORIES):
    sample = SYNTHETIC_MANIFEST[SYNTHETIC_MANIFEST.category == category].sample(
        n=min(2, int((SYNTHETIC_MANIFEST.category == category).sum())), random_state=SEED
    )
    for pair_index, (_, record) in enumerate(sample.iterrows()):
        axes[row_index, pair_index * 2].imshow(read_rgb(record.image_path))
        axes[row_index, pair_index * 2].set_title(f"{category}: {record.synthetic_type}")
        axes[row_index, pair_index * 2 + 1].imshow(read_mask(record.mask_path), cmap="gray")
        axes[row_index, pair_index * 2 + 1].set_title("Generated mask")
    for axis in axes[row_index]:
        axis.axis("off")
fig.suptitle("Synthetic anomalies used for label-free threshold calibration")
fig.tight_layout()
save_and_show_figure(fig, "synthetic_anomaly_gallery.png")

mark_stage("synthetic_calibration_generated", rows=len(SYNTHETIC_MANIFEST))


## 11 · Optional diffusion inpainting

This extension can replace generated masks with context-aware inpainted scratches, stains, or damage. It is disabled by default because model downloads, GPU memory, inference time, model licensing, and nondeterminism can make a teaching notebook fragile. Failure is caught so the deterministic procedural pipeline remains usable.

In [ ]:
from __future__ import annotations

# This extension is deliberately optional. The default calibration set remains
# deterministic procedural cut-paste/scratch/stain data so the rest of the notebook
# remains runnable on modest Colab hardware.
if not ENABLE_DIFFUSION:
    print("Diffusion inpainting is disabled. Set ENABLE_DIFFUSION=True in the configuration cell to run it.")
else:
    import subprocess
    import sys

    if not torch.cuda.is_available():
        print("Diffusion inpainting was requested without a GPU. Skipping to protect notebook reliability.")
    else:
        try:
            subprocess.check_call(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "diffusers>=0.31,<1",
                    "transformers>=4.45,<6",
                    "accelerate>=1,<2",
                    "safetensors>=0.4,<1",
                ]
            )
            from diffusers import AutoPipelineForInpainting

            model_id = "stable-diffusion-v1-5/stable-diffusion-inpainting"
            pipe = AutoPipelineForInpainting.from_pretrained(
                model_id,
                torch_dtype=torch.float16,
                variant="fp16",
            )
            pipe.enable_model_cpu_offload()
            diffusion_root = SESSION_ROOT / "diffusion_synthetic"
            diffusion_rows = []
            prompts = {
                "bottle": "a subtle manufacturing scratch or contamination mark on the bottle surface, realistic industrial photograph",
                "screw": "a subtle damaged screw thread or metal scratch, realistic industrial inspection photograph",
                "carpet": "a small stain, cut, or damaged carpet fiber region, realistic texture photograph",
            }
            for category in CATEGORIES:
                good_paths = split_manifest[
                    (split_manifest.category == category) & (split_manifest.role == "calibration_good")
                ].image_path.tolist()[:DIFFUSION_SAMPLES_PER_CATEGORY]
                category_dir = diffusion_root / category
                category_dir.mkdir(parents=True, exist_ok=True)
                for item_index, path in enumerate(good_paths):
                    original = read_rgb(path)
                    _, mask, procedural_kind = make_synthetic_defect(
                        original, category=category, seed=SEED + item_index * 1009
                    )
                    image_512 = Image.fromarray(resize_rgb(original, 512))
                    mask_512 = Image.fromarray((cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST) * 255).astype(np.uint8))
                    generator = torch.Generator(device="cpu").manual_seed(SEED + item_index)
                    generated = pipe(
                        prompt=prompts[category],
                        image=image_512,
                        mask_image=mask_512,
                        guidance_scale=7.0,
                        num_inference_steps=25,
                        generator=generator,
                    ).images[0]
                    out_path = category_dir / f"{Path(path).stem}_diffusion_{item_index:02d}.png"
                    mask_path = category_dir / f"{Path(path).stem}_diffusion_{item_index:02d}_mask.png"
                    generated.save(out_path)
                    mask_512.save(mask_path)
                    diffusion_rows.append(
                        {
                            "category": category,
                            "source_path": path,
                            "image_path": str(out_path),
                            "mask_path": str(mask_path),
                            "synthetic_type": f"diffusion_{procedural_kind}",
                            "seed": SEED + item_index,
                        }
                    )
            if diffusion_rows:
                diffusion_frame = pd.DataFrame(diffusion_rows)
                SYNTHETIC_MANIFEST = pd.concat([SYNTHETIC_MANIFEST, diffusion_frame], ignore_index=True)
                SYNTHETIC_MANIFEST.to_csv(MANIFEST_DIR / "synthetic_manifest.csv", index=False)
                copy_checkpoint(diffusion_root, "checkpoints/diffusion_synthetic")
                print(f"Added {len(diffusion_frame)} diffusion-inpainted anomalies to the calibration manifest.")
            mark_stage("diffusion_extension", state="complete", generated=len(diffusion_rows))
        except Exception as exc:
            print(f"Diffusion extension failed safely and the procedural pipeline remains available: {exc}")
            mark_stage("diffusion_extension", state="skipped_after_error", error=repr(exc))


## 12 · Shared feature-memory utilities

The next cells need consistent cache formats, pixel sampling, test ordering, and synthetic CV probes. Defining these helpers once keeps the model comparisons aligned and makes the later ensemble possible.

In [ ]:
from __future__ import annotations


def sample_map_values(maps: np.ndarray, max_values: int = 1_000_000, seed: int = SEED) -> np.ndarray:
    flat = np.asarray(maps, dtype=np.float32).reshape(-1)
    if len(flat) <= max_values:
        return flat
    rng = np.random.default_rng(seed)
    return flat[rng.choice(len(flat), size=max_values, replace=False)]


def resize_mask_list(mask_paths, output_size: int) -> list[np.ndarray]:
    masks = []
    for path in mask_paths:
        masks.append(read_mask(path if path else None, image_size=output_size))
    return masks


def selected_test_frame(category: str) -> pd.DataFrame:
    frame = split_manifest[
        (split_manifest.category == category)
        & (split_manifest.role.isin(["oracle_threshold_calibration", "operating_test"]))
    ].drop_duplicates("image_path")
    return frame.sort_values(["defect_type", "image_path"]).reset_index(drop=True)


def generate_cv_synthetics(paths: list[str], category: str) -> tuple[list[np.ndarray], list[np.ndarray]]:
    images: list[np.ndarray] = []
    masks: list[np.ndarray] = []
    for index, path in enumerate(paths):
        synthetic, mask, _ = make_synthetic_defect(
            read_rgb(path), category=category, seed=SEED + index * 7919
        )
        images.append(synthetic)
        masks.append(mask)
    return images, masks


def save_method_cache(
    path: Path,
    test_frame: pd.DataFrame,
    test_scores: np.ndarray,
    test_maps: np.ndarray,
    cal_good_paths: list[str],
    cal_good_scores: np.ndarray,
    cal_good_maps: np.ndarray,
    syn_frame: pd.DataFrame,
    syn_scores: np.ndarray,
    syn_maps: np.ndarray,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        test_paths=np.asarray(test_frame.image_path.tolist(), dtype=str),
        test_labels=np.asarray(test_frame.label.tolist(), dtype=np.int16),
        test_defect_types=np.asarray(test_frame.defect_type.tolist(), dtype=str),
        test_mask_paths=np.asarray(test_frame.mask_path.fillna("").tolist(), dtype=str),
        test_scores_raw=np.asarray(test_scores, dtype=np.float32),
        test_maps=np.asarray(test_maps, dtype=np.float16),
        cal_good_paths=np.asarray(cal_good_paths, dtype=str),
        cal_good_scores_raw=np.asarray(cal_good_scores, dtype=np.float32),
        cal_good_maps=np.asarray(cal_good_maps, dtype=np.float16),
        syn_paths=np.asarray(syn_frame.image_path.tolist(), dtype=str),
        syn_mask_paths=np.asarray(syn_frame.mask_path.tolist(), dtype=str),
        syn_scores_raw=np.asarray(syn_scores, dtype=np.float32),
        syn_maps=np.asarray(syn_maps, dtype=np.float16),
    )


def load_method_cache(category: str, method: str):
    return np.load(WORK_CACHE_DIR / method / f"{category}.npz", allow_pickle=False)


print("PatchCore, calibration, and cache helper functions are ready.")


## 13 · Train and tune the PatchCore-style detector

Multi-scale MobileNetV3-Small patch embeddings are extracted from good images. A good-only K-fold search evaluates memory-bank ratio, top-pixel aggregation, and smoothing using validation-good images plus deterministic synthetic counterparts. The objective balances image detection, pixel localization, and memory footprint.

The selected model is fitted on model-training good images only. Held-out good and synthetic images calibrate the 0–100 score scale and cost-sensitive threshold. Test maps are cached for later evaluation.

In [ ]:
from __future__ import annotations

PATCH_CACHE_DIR = WORK_CACHE_DIR / "patchcore"
PATCH_CACHE_DIR.mkdir(parents=True, exist_ok=True)
PATCH_HPO_ROWS: list[dict] = []

extractor = MobileNetPatchExtractor(pretrained=True).to(DEVICE).eval()
patch_size = int(CFG["patch_image_size"])
feature_batch_size = int(CFG["feature_batch_size"])

for category in CATEGORIES:
    print("\n" + "=" * 88)
    print(f"PATCHCORE-STYLE TRAINING: {category.upper()}")
    print("=" * 88)
    artifact_folder = ARTIFACT_DIR / category / "patchcore"
    artifact_folder.mkdir(parents=True, exist_ok=True)
    cache_path = PATCH_CACHE_DIR / f"{category}.npz"

    drive_artifact = DRIVE_RUN_DIR / "artifacts" / category / "patchcore"
    drive_cache = DRIVE_RUN_DIR / "checkpoints" / "patchcore" / f"{category}.npz"
    if RESUME_RUN_ID and not FORCE_RETRAIN and drive_artifact.exists() and drive_cache.exists():
        print(f"Restoring completed PatchCore artifacts for {category} from Drive.")
        if artifact_folder.exists():
            shutil.rmtree(artifact_folder)
        shutil.copytree(drive_artifact, artifact_folder)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_cache, cache_path)
        hpo_path = artifact_folder / "hpo.csv"
        if hpo_path.exists():
            PATCH_HPO_ROWS.extend(pd.read_csv(hpo_path).to_dict("records"))
        continue

    train_paths = split_manifest[
        (split_manifest.category == category) & (split_manifest.role == "model_train_good")
    ].image_path.tolist()
    cal_good_paths = split_manifest[
        (split_manifest.category == category) & (split_manifest.role == "calibration_good")
    ].image_path.tolist()
    syn_frame = SYNTHETIC_MANIFEST[SYNTHETIC_MANIFEST.category == category].reset_index(drop=True)
    test_frame = selected_test_frame(category)

    # Inner good-only CV: validation images are paired with deterministic synthetic counterparts.
    rng = np.random.default_rng(SEED)
    cv_paths = train_paths.copy()
    if len(cv_paths) > CFG["cv_cap"]:
        selected = rng.choice(len(cv_paths), size=CFG["cv_cap"], replace=False)
        cv_paths = [cv_paths[int(i)] for i in selected]
    cv_synthetic_images, cv_synthetic_masks = generate_cv_synthetics(cv_paths, category)

    print(f"Extracting CV features for {len(cv_paths)} good and {len(cv_synthetic_images)} synthetic images...")
    cv_good_maps, _ = extract_patch_maps(
        extractor, cv_paths, patch_size, feature_batch_size, DEVICE
    )
    cv_syn_maps, _ = extract_patch_maps(
        extractor, cv_synthetic_images, patch_size, feature_batch_size, DEVICE
    )

    n_splits = max(2, min(int(CFG["cv_folds"]), len(cv_paths)))
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    candidate_rows = []
    candidate_grid = list(
        itertools.product(CFG["memory_ratios"], CFG["topk_fractions"], CFG["blur_sigmas"])
    )
    print(f"Grid-searching {len(candidate_grid)} PatchCore configurations over {n_splits} folds...")

    for ratio, topk_fraction, blur_sigma in candidate_grid:
        fold_rows = []
        for fold, (train_indices, val_indices) in enumerate(kfold.split(cv_paths), start=1):
            bank = build_memory_bank(
                cv_good_maps[train_indices],
                ratio=float(ratio),
                max_mb=float(CFG["max_memory_mb"]),
                seed=SEED + fold,
            )
            scoring_started = time.perf_counter()
            good_scores, good_maps = score_patch_maps(
                cv_good_maps[val_indices],
                bank,
                output_size=patch_size,
                topk_fraction=float(topk_fraction),
                blur_sigma=float(blur_sigma),
                device="cpu",
                backend="auto",
            )
            syn_scores, syn_maps = score_patch_maps(
                cv_syn_maps[val_indices],
                bank,
                output_size=patch_size,
                topk_fraction=float(topk_fraction),
                blur_sigma=float(blur_sigma),
                device="cpu",
                backend="auto",
            )
            scoring_ms_per_image = 1000.0 * (time.perf_counter() - scoring_started) / max(
                1, len(good_scores) + len(syn_scores)
            )
            labels = np.concatenate([np.zeros(len(good_scores)), np.ones(len(syn_scores))])
            scores = np.concatenate([good_scores, syn_scores])
            masks = [np.zeros((patch_size, patch_size), dtype=np.uint8) for _ in good_scores]
            masks.extend(
                cv2.resize(cv_synthetic_masks[int(i)], (patch_size, patch_size), interpolation=cv2.INTER_NEAREST)
                for i in val_indices
            )
            maps = [*good_maps, *syn_maps]
            image_auc = safe_roc_auc(labels, scores)
            pixel_auc = sampled_pixel_auroc(maps, masks, max_pixels=min(300_000, CFG["pixel_max_samples"]), seed=SEED)
            memory_mb = bank.nbytes / 1024**2
            fold_rows.append(
                {
                    "image_auroc_synthetic": image_auc,
                    "pixel_auroc_synthetic": pixel_auc,
                    "memory_mb": memory_mb,
                    "scoring_ms_per_image": scoring_ms_per_image,
                }
            )
            del bank, good_maps, syn_maps

        fold_frame = pd.DataFrame(fold_rows)
        image_auc = float(fold_frame.image_auroc_synthetic.mean())
        pixel_auc = float(fold_frame.pixel_auroc_synthetic.mean())
        memory_mb = float(fold_frame.memory_mb.mean())
        scoring_ms_per_image = float(fold_frame.scoring_ms_per_image.mean())
        objective = (
            0.60 * np.nan_to_num(image_auc)
            + 0.38 * np.nan_to_num(pixel_auc)
            - 0.002 * np.log1p(memory_mb)
            - 0.001 * np.log1p(scoring_ms_per_image)
        )
        candidate_rows.append(
            {
                "category": category,
                "memory_ratio": float(ratio),
                "topk_fraction": float(topk_fraction),
                "blur_sigma": float(blur_sigma),
                "cv_folds": n_splits,
                "image_auroc_synthetic": image_auc,
                "pixel_auroc_synthetic": pixel_auc,
                "memory_mb": memory_mb,
                "scoring_ms_per_image": scoring_ms_per_image,
                "objective": float(objective),
            }
        )

    hpo_frame = pd.DataFrame(candidate_rows).sort_values(
        ["objective", "image_auroc_synthetic", "pixel_auroc_synthetic"], ascending=False
    ).reset_index(drop=True)
    best = hpo_frame.iloc[0].to_dict()
    hpo_frame.to_csv(artifact_folder / "hpo.csv", index=False)
    PATCH_HPO_ROWS.extend(hpo_frame.to_dict("records"))
    print("Selected parameters:")
    print(pd.Series(best).to_string())

    del cv_good_maps, cv_syn_maps, cv_synthetic_images, cv_synthetic_masks
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Final fit on model-training good images only.
    train_maps, train_global = extract_patch_maps(
        extractor, train_paths, patch_size, feature_batch_size, DEVICE
    )
    memory_bank = build_memory_bank(
        train_maps,
        ratio=float(best["memory_ratio"]),
        max_mb=float(CFG["max_memory_mb"]),
        seed=SEED,
    )
    np.save(artifact_folder / "memory_bank.npy", memory_bank)

    cal_good_maps_features, _ = extract_patch_maps(
        extractor, cal_good_paths, patch_size, feature_batch_size, DEVICE
    )
    syn_maps_features, _ = extract_patch_maps(
        extractor, syn_frame.image_path.tolist(), patch_size, feature_batch_size, DEVICE
    )
    test_maps_features, _ = extract_patch_maps(
        extractor, test_frame.image_path.tolist(), patch_size, feature_batch_size, DEVICE
    )

    score_kwargs = {
        "memory_bank": memory_bank,
        "output_size": patch_size,
        "topk_fraction": float(best["topk_fraction"]),
        "blur_sigma": float(best["blur_sigma"]),
        "device": "cpu",
        "backend": "auto",
    }
    cal_good_scores, cal_good_maps = score_patch_maps(cal_good_maps_features, **score_kwargs)
    syn_scores, syn_maps = score_patch_maps(syn_maps_features, **score_kwargs)
    test_scores, test_maps = score_patch_maps(test_maps_features, **score_kwargs)

    score_scaler = robust_scaler(np.concatenate([cal_good_scores, syn_scores]))
    pixel_scaler = robust_scaler(
        np.concatenate(
            [sample_map_values(cal_good_maps, 500_000), sample_map_values(syn_maps, 500_000)]
        )
    )
    calibration_scores = normalize_with_scaler(
        np.concatenate([cal_good_scores, syn_scores]), score_scaler
    )
    calibration_labels = np.concatenate(
        [np.zeros(len(cal_good_scores), dtype=np.int16), np.ones(len(syn_scores), dtype=np.int16)]
    )
    synthetic_threshold, threshold_curve = choose_cost_threshold(
        calibration_scores,
        calibration_labels,
        defect_prevalence=DEFECT_PREVALENCE,
        cost_escape=COST_ESCAPE,
        cost_false_reject=COST_FALSE_REJECT,
        check_band=CHECK_BAND,
        cost_review=COST_HUMAN_REVIEW,
    )
    threshold_curve = threshold_curve.rename(columns={"threshold": "threshold_normalized"})
    threshold_curve.to_csv(artifact_folder / "threshold_curve.csv", index=False)

    config = {
        "category": category,
        "image_size": patch_size,
        "memory_ratio": float(best["memory_ratio"]),
        "topk_fraction": float(best["topk_fraction"]),
        "blur_sigma": float(best["blur_sigma"]),
        "feature_extractor": "MobileNetV3-Small blocks 3, 8, 11",
        "memory_dtype": "float16",
    }
    calibration = {
        "threshold_source": "held-out good plus synthetic anomalies; no real defects",
        "default_threshold_normalized": float(synthetic_threshold),
        "synthetic_threshold_normalized": float(synthetic_threshold),
        "check_band": CHECK_BAND,
        "score_scaler": score_scaler,
        "pixel_scaler": pixel_scaler,
        "output_size": patch_size,
        "calibration_good_count": len(cal_good_scores),
        "synthetic_count": len(syn_scores),
        "memory_bank_mb": memory_bank.nbytes / 1024**2,
        "global_reference_mean": train_global.mean(axis=0).tolist(),
        "global_reference_std": np.maximum(train_global.std(axis=0), 1e-4).tolist(),
    }
    save_json(config, artifact_folder / "config.json")
    save_json(calibration, artifact_folder / "calibration.json")

    save_method_cache(
        cache_path,
        test_frame,
        test_scores,
        test_maps,
        cal_good_paths,
        cal_good_scores,
        cal_good_maps,
        syn_frame,
        syn_scores,
        syn_maps,
    )

    copy_checkpoint(artifact_folder, Path("artifacts") / category / "patchcore")
    copy_checkpoint(cache_path, Path("checkpoints") / "patchcore" / cache_path.name)
    mark_stage(
        f"patchcore_{category}",
        memory_bank_mb=round(memory_bank.nbytes / 1024**2, 3),
        synthetic_threshold=round(float(synthetic_threshold), 3),
        best_parameters=config,
    )

    del (
        train_maps,
        train_global,
        memory_bank,
        cal_good_maps_features,
        syn_maps_features,
        test_maps_features,
        cal_good_maps,
        syn_maps,
        test_maps,
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

patch_hpo_all = pd.DataFrame(PATCH_HPO_ROWS)
patch_hpo_all.to_csv(REPORT_DIR / "patchcore_hpo.csv", index=False)
copy_checkpoint(REPORT_DIR / "patchcore_hpo.csv", "reports/patchcore_hpo.csv")
mark_stage("patchcore_all_categories", rows=len(patch_hpo_all))


## 14 · Fit the aligned mean-good-image baseline

This transparent classical baseline aligns images to a robust good reference, computes pixel-wise mean and standard deviation, and scores standardized differences. Alignment mode and variance floor are tuned against held-out good and synthetic images. It provides an important sanity check: in a highly controlled station it may be competitive, while pose and texture variation reveal its limits.

In [ ]:
from __future__ import annotations

MEAN_CACHE_DIR = WORK_CACHE_DIR / "mean"
MEAN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
MEAN_HPO_ROWS: list[dict] = []

alignment_candidates = {
    "bottle": ["euclidean"] if PROFILE == "smoke" else ["euclidean", "affine"],
    "screw": ["affine"] if PROFILE == "smoke" else ["none", "affine"],
    "carpet": ["none"] if PROFILE == "smoke" else ["none", "translation"],
}
std_floor_candidates = [6.0] if PROFILE == "smoke" else ([4.0, 8.0] if PROFILE == "balanced" else [4.0, 6.0, 10.0])

for category in CATEGORIES:
    print("\n" + "=" * 88)
    print(f"ALIGNED MEAN-IMAGE BASELINE: {category.upper()}")
    print("=" * 88)
    artifact_folder = ARTIFACT_DIR / category / "mean"
    artifact_folder.mkdir(parents=True, exist_ok=True)
    cache_path = MEAN_CACHE_DIR / f"{category}.npz"

    drive_artifact = DRIVE_RUN_DIR / "artifacts" / category / "mean"
    drive_cache = DRIVE_RUN_DIR / "checkpoints" / "mean" / f"{category}.npz"
    if RESUME_RUN_ID and not FORCE_RETRAIN and drive_artifact.exists() and drive_cache.exists():
        print(f"Restoring completed mean-baseline artifacts for {category} from Drive.")
        if artifact_folder.exists():
            shutil.rmtree(artifact_folder)
        shutil.copytree(drive_artifact, artifact_folder)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_cache, cache_path)
        hpo_path = artifact_folder / "hpo.csv"
        if hpo_path.exists():
            MEAN_HPO_ROWS.extend(pd.read_csv(hpo_path).to_dict("records"))
        continue

    train_paths = split_manifest[
        (split_manifest.category == category) & (split_manifest.role == "model_train_good")
    ].image_path.tolist()
    cal_good_paths = split_manifest[
        (split_manifest.category == category) & (split_manifest.role == "calibration_good")
    ].image_path.tolist()
    syn_frame = SYNTHETIC_MANIFEST[SYNTHETIC_MANIFEST.category == category].reset_index(drop=True)
    test_frame = selected_test_frame(category)

    # Keep the good calibration pool untouched for final score scaling and threshold selection.
    # Hyperparameters are selected with an inner split of the model-training good images.
    mean_fit_paths, mean_hpo_val_paths = train_test_split(
        train_paths, test_size=0.20, random_state=SEED, shuffle=True
    )
    mean_hpo_syn_images, mean_hpo_syn_masks = generate_cv_synthetics(mean_hpo_val_paths, category)

    candidate_rows = []
    for alignment_mode, std_floor in itertools.product(alignment_candidates[category], std_floor_candidates):
        candidate = MeanImageBaseline(
            MeanBaselineConfig(
                image_size=int(CFG["patch_image_size"]),
                alignment_mode=alignment_mode,
                topk_fraction=0.01,
                blur_sigma=3.0,
                std_floor=float(std_floor),
            )
        ).fit(mean_fit_paths)

        good_predictions = [candidate.predict(path) for path in mean_hpo_val_paths]
        syn_predictions = [candidate.predict(image) for image in mean_hpo_syn_images]
        good_scores = np.asarray([item[0] for item in good_predictions], dtype=np.float32)
        syn_scores = np.asarray([item[0] for item in syn_predictions], dtype=np.float32)
        good_maps = np.stack([item[1] for item in good_predictions])
        syn_maps = np.stack([item[1] for item in syn_predictions])
        labels = np.concatenate([np.zeros(len(good_scores)), np.ones(len(syn_scores))])
        scores = np.concatenate([good_scores, syn_scores])
        masks = [np.zeros_like(good_maps[0], dtype=np.uint8) for _ in good_maps]
        masks.extend(
            cv2.resize(mask, (int(CFG["patch_image_size"]), int(CFG["patch_image_size"])), interpolation=cv2.INTER_NEAREST)
            for mask in mean_hpo_syn_masks
        )
        image_auc = safe_roc_auc(labels, scores)
        pixel_auc = sampled_pixel_auroc(
            [*good_maps, *syn_maps], masks, max_pixels=min(400_000, CFG["pixel_max_samples"]), seed=SEED
        )
        finite_corr = np.asarray(candidate.alignment_correlations, dtype=np.float32)
        finite_corr = finite_corr[np.isfinite(finite_corr)]
        median_correlation = float(np.median(finite_corr)) if len(finite_corr) else float("nan")
        objective = 0.60 * np.nan_to_num(image_auc) + 0.40 * np.nan_to_num(pixel_auc)
        candidate_rows.append(
            {
                "category": category,
                "alignment_mode": alignment_mode,
                "std_floor": float(std_floor),
                "image_auroc_synthetic": image_auc,
                "pixel_auroc_synthetic": pixel_auc,
                "median_alignment_correlation": median_correlation,
                "objective": float(objective),
            }
        )
        del candidate, good_predictions, syn_predictions, good_maps, syn_maps
        gc.collect()

    hpo_frame = pd.DataFrame(candidate_rows).sort_values("objective", ascending=False).reset_index(drop=True)
    best = hpo_frame.iloc[0].to_dict()
    hpo_frame.to_csv(artifact_folder / "hpo.csv", index=False)
    MEAN_HPO_ROWS.extend(hpo_frame.to_dict("records"))
    print(hpo_frame.to_string(index=False))

    model = MeanImageBaseline(
        MeanBaselineConfig(
            image_size=int(CFG["patch_image_size"]),
            alignment_mode=str(best["alignment_mode"]),
            topk_fraction=0.01,
            blur_sigma=3.0,
            std_floor=float(best["std_floor"]),
        )
    ).fit(train_paths)
    model.save(artifact_folder)

    def score_items(items):
        predictions = [model.predict(item) for item in items]
        return (
            np.asarray([prediction[0] for prediction in predictions], dtype=np.float32),
            np.stack([prediction[1] for prediction in predictions]).astype(np.float32),
        )

    cal_good_scores, cal_good_maps = score_items(cal_good_paths)
    syn_scores, syn_maps = score_items(syn_frame.image_path.tolist())
    test_scores, test_maps = score_items(test_frame.image_path.tolist())

    score_scaler = robust_scaler(np.concatenate([cal_good_scores, syn_scores]))
    pixel_scaler = robust_scaler(
        np.concatenate([sample_map_values(cal_good_maps, 500_000), sample_map_values(syn_maps, 500_000)])
    )
    normalized_calibration = normalize_with_scaler(np.concatenate([cal_good_scores, syn_scores]), score_scaler)
    calibration_labels = np.concatenate(
        [np.zeros(len(cal_good_scores), dtype=np.int16), np.ones(len(syn_scores), dtype=np.int16)]
    )
    synthetic_threshold, threshold_curve = choose_cost_threshold(
        normalized_calibration,
        calibration_labels,
        defect_prevalence=DEFECT_PREVALENCE,
        cost_escape=COST_ESCAPE,
        cost_false_reject=COST_FALSE_REJECT,
        check_band=CHECK_BAND,
        cost_review=COST_HUMAN_REVIEW,
    )
    threshold_curve.rename(columns={"threshold": "threshold_normalized"}).to_csv(
        artifact_folder / "threshold_curve.csv", index=False
    )
    calibration = {
        "threshold_source": "held-out good plus synthetic anomalies; no real defects",
        "default_threshold_normalized": float(synthetic_threshold),
        "synthetic_threshold_normalized": float(synthetic_threshold),
        "check_band": CHECK_BAND,
        "score_scaler": score_scaler,
        "pixel_scaler": pixel_scaler,
        "output_size": int(CFG["patch_image_size"]),
        "alignment_mode": best["alignment_mode"],
        "std_floor": float(best["std_floor"]),
        "calibration_good_count": len(cal_good_scores),
        "synthetic_count": len(syn_scores),
    }
    save_json(calibration, artifact_folder / "calibration.json")

    save_method_cache(
        cache_path,
        test_frame,
        test_scores,
        test_maps,
        cal_good_paths,
        cal_good_scores,
        cal_good_maps,
        syn_frame,
        syn_scores,
        syn_maps,
    )
    copy_checkpoint(artifact_folder, Path("artifacts") / category / "mean")
    copy_checkpoint(cache_path, Path("checkpoints") / "mean" / cache_path.name)
    mark_stage(
        f"mean_{category}",
        synthetic_threshold=round(float(synthetic_threshold), 3),
        alignment_mode=best["alignment_mode"],
        std_floor=float(best["std_floor"]),
    )
    del model, cal_good_maps, syn_maps, test_maps
    gc.collect()

mean_hpo_all = pd.DataFrame(MEAN_HPO_ROWS)
mean_hpo_all.to_csv(REPORT_DIR / "mean_baseline_hpo.csv", index=False)
copy_checkpoint(REPORT_DIR / "mean_baseline_hpo.csv", "reports/mean_baseline_hpo.csv")
mark_stage("mean_all_categories", rows=len(mean_hpo_all))


## 15 · Tune and train the convolutional VAE

Candidate VAEs vary latent size, channel width, KL weight, and learning rate. A successive-halving-style short search uses an inner good validation split and synthetic probes. The selected VAE is then trained with early stopping. Training history and the current best checkpoint are copied to Drive during training.

The anomaly signal combines RGB reconstruction error with gradient error so thin scratches and boundaries receive more weight.

In [ ]:
from __future__ import annotations

VAE_CACHE_DIR = WORK_CACHE_DIR / "vae"
VAE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
VAE_HPO_ROWS: list[dict] = []


def score_vae_items(model: ConvVAE, items, output_size: int):
    scores = []
    maps = []
    reconstructions = []
    for item in items:
        score, anomaly_map, reconstruction = vae_predict(model, item, device=DEVICE, output_size=output_size)
        scores.append(score)
        maps.append(anomaly_map)
        reconstructions.append(reconstruction)
    return np.asarray(scores, dtype=np.float32), np.stack(maps).astype(np.float32), reconstructions


for category in CATEGORIES:
    print("\n" + "=" * 88)
    print(f"CONVOLUTIONAL VAE: {category.upper()}")
    print("=" * 88)
    artifact_folder = ARTIFACT_DIR / category / "vae"
    artifact_folder.mkdir(parents=True, exist_ok=True)
    cache_path = VAE_CACHE_DIR / f"{category}.npz"

    drive_artifact = DRIVE_RUN_DIR / "artifacts" / category / "vae"
    drive_cache = DRIVE_RUN_DIR / "checkpoints" / "vae" / f"{category}.npz"
    if RESUME_RUN_ID and not FORCE_RETRAIN and drive_artifact.exists() and drive_cache.exists():
        print(f"Restoring completed VAE artifacts for {category} from Drive.")
        if artifact_folder.exists():
            shutil.rmtree(artifact_folder)
        shutil.copytree(drive_artifact, artifact_folder)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_cache, cache_path)
        hpo_path = artifact_folder / "hpo.csv"
        if hpo_path.exists():
            VAE_HPO_ROWS.extend(pd.read_csv(hpo_path).to_dict("records"))
        continue

    model_train_paths = split_manifest[
        (split_manifest.category == category) & (split_manifest.role == "model_train_good")
    ].image_path.tolist()
    cal_good_paths = split_manifest[
        (split_manifest.category == category) & (split_manifest.role == "calibration_good")
    ].image_path.tolist()
    syn_frame = SYNTHETIC_MANIFEST[SYNTHETIC_MANIFEST.category == category].reset_index(drop=True)
    test_frame = selected_test_frame(category)

    inner_train_paths, inner_val_paths = train_test_split(
        model_train_paths, test_size=0.15, random_state=SEED, shuffle=True
    )
    validation_probe_paths = inner_val_paths[: min(30, len(inner_val_paths))]
    validation_synthetic_images, validation_synthetic_masks = generate_cv_synthetics(
        validation_probe_paths, category
    )

    num_workers = 2 if IN_COLAB else 0
    train_dataset = RGBImageDataset(
        inner_train_paths, image_size=int(CFG["vae_image_size"]), augment=True, category=category
    )
    val_dataset = RGBImageDataset(
        inner_val_paths, image_size=int(CFG["vae_image_size"]), augment=False, category=category
    )
    loader_kwargs = {
        "batch_size": int(CFG["vae_batch_size"]),
        "num_workers": num_workers,
        "pin_memory": torch.cuda.is_available(),
    }
    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)

    candidate_rows = []
    for candidate_index, candidate_dict in enumerate(CFG["vae_candidates"]):
        config = VAEConfig(
            image_size=int(CFG["vae_image_size"]),
            latent_dim=int(candidate_dict["latent_dim"]),
            base_channels=int(candidate_dict["base_channels"]),
            beta=float(candidate_dict["beta"]),
            learning_rate=float(candidate_dict["learning_rate"]),
            topk_fraction=0.01,
            blur_sigma=2.5,
        )
        model = ConvVAE(config)
        checkpoint_path = WORK_CACHE_DIR / "vae_search" / category / f"candidate_{candidate_index:02d}.pt"
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        history = train_vae(
            model,
            train_loader,
            val_loader,
            epochs=int(CFG["vae_search_epochs"]),
            device=DEVICE,
            checkpoint_path=checkpoint_path,
            patience=max(2, int(CFG["vae_search_epochs"])),
        )
        probe_good_scores, probe_good_maps, _ = score_vae_items(
            model, validation_probe_paths, int(CFG["patch_image_size"])
        )
        probe_syn_scores, probe_syn_maps, _ = score_vae_items(
            model, validation_synthetic_images, int(CFG["patch_image_size"])
        )
        labels = np.concatenate([np.zeros(len(probe_good_scores)), np.ones(len(probe_syn_scores))])
        scores = np.concatenate([probe_good_scores, probe_syn_scores])
        masks = [np.zeros_like(probe_good_maps[0], dtype=np.uint8) for _ in probe_good_maps]
        masks.extend(
            cv2.resize(mask, (int(CFG["patch_image_size"]), int(CFG["patch_image_size"])), interpolation=cv2.INTER_NEAREST)
            for mask in validation_synthetic_masks
        )
        image_auc = safe_roc_auc(labels, scores)
        pixel_auc = sampled_pixel_auroc(
            [*probe_good_maps, *probe_syn_maps], masks, max_pixels=min(300_000, CFG["pixel_max_samples"]), seed=SEED
        )
        val_loss = float(min(row["val_loss"] for row in history))
        objective = 0.62 * np.nan_to_num(image_auc) + 0.36 * np.nan_to_num(pixel_auc) - 0.02 * val_loss
        candidate_rows.append(
            {
                "category": category,
                **candidate_dict,
                "search_epochs": len(history),
                "best_val_loss": val_loss,
                "image_auroc_synthetic": image_auc,
                "pixel_auroc_synthetic": pixel_auc,
                "objective": float(objective),
            }
        )
        del model, probe_good_maps, probe_syn_maps
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    hpo_frame = pd.DataFrame(candidate_rows).sort_values("objective", ascending=False).reset_index(drop=True)
    best = hpo_frame.iloc[0].to_dict()
    hpo_frame.to_csv(artifact_folder / "hpo.csv", index=False)
    VAE_HPO_ROWS.extend(hpo_frame.to_dict("records"))
    print(hpo_frame.to_string(index=False))

    final_config = VAEConfig(
        image_size=int(CFG["vae_image_size"]),
        latent_dim=int(best["latent_dim"]),
        base_channels=int(best["base_channels"]),
        beta=float(best["beta"]),
        learning_rate=float(best["learning_rate"]),
        topk_fraction=0.01,
        blur_sigma=2.5,
    )
    final_model = ConvVAE(final_config)
    final_checkpoint = artifact_folder / "model.pt"
    drive_epoch_dir = DRIVE_RUN_DIR / "checkpoints" / "vae_training" / category
    drive_epoch_dir.mkdir(parents=True, exist_ok=True)
    resume_training = False
    drive_best_checkpoint = drive_epoch_dir / "best_model.pt"
    if RESUME_RUN_ID and drive_best_checkpoint.exists():
        resume_payload = torch.load(drive_best_checkpoint, map_location="cpu", weights_only=False)
        if resume_payload.get("config") == final_config.__dict__:
            shutil.copy2(drive_best_checkpoint, final_checkpoint)
            resume_training = True
            print(f"Warm-resuming {category} VAE from Drive checkpoint at epoch {resume_payload.get('epoch')}.")
        else:
            print("Stored VAE checkpoint configuration differs from the selected configuration; starting cleanly.")

    history_rows: list[dict] = []
    drive_history_path = drive_epoch_dir / "training_history.csv"
    if resume_training and drive_history_path.exists():
        history_rows = pd.read_csv(drive_history_path).to_dict("records")

    def epoch_checkpoint(row: dict) -> None:
        history_rows.append(dict(row))
        pd.DataFrame(history_rows).to_csv(artifact_folder / "training_history.csv", index=False)
        shutil.copy2(artifact_folder / "training_history.csv", drive_epoch_dir / "training_history.csv")
        if row.get("improved") and final_checkpoint.exists():
            shutil.copy2(final_checkpoint, drive_epoch_dir / "best_model.pt")
        print(
            f"{category} epoch {int(row['epoch']):02d}: train={row['train_loss']:.5f}, "
            f"val={row['val_loss']:.5f}, improved={bool(row.get('improved'))}"
        )

    final_history = train_vae(
        final_model,
        train_loader,
        val_loader,
        epochs=int(CFG["vae_final_epochs"]),
        device=DEVICE,
        checkpoint_path=final_checkpoint,
        patience=6 if PROFILE != "smoke" else 3,
        progress_callback=epoch_checkpoint,
        resume=resume_training,
    )
    final_model = load_vae(final_checkpoint, device=DEVICE)

    cal_good_scores, cal_good_maps, _ = score_vae_items(
        final_model, cal_good_paths, int(CFG["patch_image_size"])
    )
    syn_scores, syn_maps, _ = score_vae_items(
        final_model, syn_frame.image_path.tolist(), int(CFG["patch_image_size"])
    )
    test_scores, test_maps, _ = score_vae_items(
        final_model, test_frame.image_path.tolist(), int(CFG["patch_image_size"])
    )

    score_scaler = robust_scaler(np.concatenate([cal_good_scores, syn_scores]))
    pixel_scaler = robust_scaler(
        np.concatenate([sample_map_values(cal_good_maps, 500_000), sample_map_values(syn_maps, 500_000)])
    )
    normalized_calibration = normalize_with_scaler(np.concatenate([cal_good_scores, syn_scores]), score_scaler)
    calibration_labels = np.concatenate(
        [np.zeros(len(cal_good_scores), dtype=np.int16), np.ones(len(syn_scores), dtype=np.int16)]
    )
    synthetic_threshold, threshold_curve = choose_cost_threshold(
        normalized_calibration,
        calibration_labels,
        defect_prevalence=DEFECT_PREVALENCE,
        cost_escape=COST_ESCAPE,
        cost_false_reject=COST_FALSE_REJECT,
        check_band=CHECK_BAND,
        cost_review=COST_HUMAN_REVIEW,
    )
    threshold_curve.rename(columns={"threshold": "threshold_normalized"}).to_csv(
        artifact_folder / "threshold_curve.csv", index=False
    )
    calibration = {
        "threshold_source": "held-out good plus synthetic anomalies; no real defects",
        "default_threshold_normalized": float(synthetic_threshold),
        "synthetic_threshold_normalized": float(synthetic_threshold),
        "check_band": CHECK_BAND,
        "score_scaler": score_scaler,
        "pixel_scaler": pixel_scaler,
        "output_size": int(CFG["patch_image_size"]),
        "vae_image_size": int(CFG["vae_image_size"]),
        "latent_dim": int(best["latent_dim"]),
        "base_channels": int(best["base_channels"]),
        "beta": float(best["beta"]),
        "learning_rate": float(best["learning_rate"]),
        "calibration_good_count": len(cal_good_scores),
        "synthetic_count": len(syn_scores),
        "trained_epochs": len(final_history),
    }
    save_json(calibration, artifact_folder / "calibration.json")

    save_method_cache(
        cache_path,
        test_frame,
        test_scores,
        test_maps,
        cal_good_paths,
        cal_good_scores,
        cal_good_maps,
        syn_frame,
        syn_scores,
        syn_maps,
    )
    copy_checkpoint(artifact_folder, Path("artifacts") / category / "vae")
    copy_checkpoint(cache_path, Path("checkpoints") / "vae" / cache_path.name)
    mark_stage(
        f"vae_{category}",
        synthetic_threshold=round(float(synthetic_threshold), 3),
        latent_dim=int(best["latent_dim"]),
        base_channels=int(best["base_channels"]),
        trained_epochs=len(final_history),
    )
    del final_model, cal_good_maps, syn_maps, test_maps
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

vae_hpo_all = pd.DataFrame(VAE_HPO_ROWS)
vae_hpo_all.to_csv(REPORT_DIR / "vae_hpo.csv", index=False)
copy_checkpoint(REPORT_DIR / "vae_hpo.csv", "reports/vae_hpo.csv")
mark_stage("vae_all_categories", rows=len(vae_hpo_all))


## 16 · Export and verify the ONNX feature extractor

The frozen feature extractor is exported once and shared across categories. The cell measures its actual size and compares ONNX Runtime outputs with PyTorch outputs. Packaging stops if numerical verification fails.

In [ ]:
from __future__ import annotations

import json

onnx_folder = ARTIFACT_DIR / "onnx"
onnx_folder.mkdir(parents=True, exist_ok=True)
onnx_path = onnx_folder / "mobilenet_v3_small_patch.onnx"
onnx_report_path = onnx_folder / "export_report.json"

if RESUME_RUN_ID and not FORCE_REBUILD_ONNX and not onnx_path.exists():
    restore_checkpoint("artifacts/onnx", onnx_folder)

reuse_onnx = False
if not FORCE_REBUILD_ONNX and onnx_path.exists() and onnx_report_path.exists():
    try:
        existing_report = json.loads(onnx_report_path.read_text(encoding="utf-8"))
        reuse_onnx = bool(existing_report.get("verified")) and onnx_path.stat().st_size > 0
    except Exception:
        reuse_onnx = False

if reuse_onnx:
    onnx_result = existing_report
    print(f"Reusing verified ONNX model: {onnx_path}")
else:
    export_model = extractor.to("cpu").eval() if "extractor" in globals() else MobileNetPatchExtractor(pretrained=True).cpu().eval()
    onnx_result = export_feature_extractor_onnx(
        export_model,
        onnx_path,
        image_size=int(CFG["patch_image_size"]),
        verify=True,
    )
    save_json(onnx_result, onnx_report_path)

copy_checkpoint(onnx_folder, "artifacts/onnx")
print(json.dumps(onnx_result, indent=2))
if not onnx_result.get("verified", False):
    raise RuntimeError(f"ONNX verification failed: {onnx_result}")
mark_stage("onnx_export_verified", **onnx_result)


## 17 · Build the ensemble and evaluate all methods

PatchCore and VAE scores are calibrated before a synthetic-only weight search builds the optional ensemble. Every method is then evaluated on real MVTec test images with:

- image AUROC and average precision;
- bootstrap AUROC confidence intervals;
- pixel AUROC and AUPRO@0.30;
- synthetic-threshold and real-label-oracle operating metrics;
- false rejects, escapes, human-review rate, and expected cost;
- per-defect-type escape analysis;
- qualitative panels for strong detections, escape risks, and false-reject risks.

The default packaged threshold remains synthetic-only.

In [ ]:
from __future__ import annotations

from defectspotter.evaluation import decision_from_score

ENSEMBLE_CACHE_DIR = WORK_CACHE_DIR / "ensemble"
ENSEMBLE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_HPO_ROWS: list[dict] = []

# ------------------------- Build the calibrated ensemble -------------------------
for category in CATEGORIES:
    patch_cache = load_method_cache(category, "patchcore")
    vae_cache = load_method_cache(category, "vae")
    patch_cal = json.loads((ARTIFACT_DIR / category / "patchcore" / "calibration.json").read_text())
    vae_cal = json.loads((ARTIFACT_DIR / category / "vae" / "calibration.json").read_text())

    patch_good_scores = normalize_with_scaler(patch_cache["cal_good_scores_raw"], patch_cal["score_scaler"])
    vae_good_scores = normalize_with_scaler(vae_cache["cal_good_scores_raw"], vae_cal["score_scaler"])
    patch_syn_scores = normalize_with_scaler(patch_cache["syn_scores_raw"], patch_cal["score_scaler"])
    vae_syn_scores = normalize_with_scaler(vae_cache["syn_scores_raw"], vae_cal["score_scaler"])
    patch_good_maps = normalize_with_scaler(patch_cache["cal_good_maps"].astype(np.float32), patch_cal["pixel_scaler"])
    vae_good_maps = normalize_with_scaler(vae_cache["cal_good_maps"].astype(np.float32), vae_cal["pixel_scaler"])
    patch_syn_maps = normalize_with_scaler(patch_cache["syn_maps"].astype(np.float32), patch_cal["pixel_scaler"])
    vae_syn_maps = normalize_with_scaler(vae_cache["syn_maps"].astype(np.float32), vae_cal["pixel_scaler"])

    weight_grid = [0.5] if PROFILE == "smoke" else ([0.25, 0.5, 0.75] if PROFILE == "balanced" else [0.0, 0.25, 0.5, 0.75, 1.0])
    weight_rows = []
    syn_masks = [read_mask(path, image_size=int(CFG["patch_image_size"])) for path in patch_cache["syn_mask_paths"]]
    good_zero_masks = [np.zeros((int(CFG["patch_image_size"]), int(CFG["patch_image_size"])), dtype=np.uint8) for _ in patch_good_scores]

    for weight in weight_grid:
        good_scores = weight * patch_good_scores + (1.0 - weight) * vae_good_scores
        syn_scores = weight * patch_syn_scores + (1.0 - weight) * vae_syn_scores
        good_maps = weight * patch_good_maps + (1.0 - weight) * vae_good_maps
        syn_maps = weight * patch_syn_maps + (1.0 - weight) * vae_syn_maps
        labels = np.concatenate([np.zeros(len(good_scores)), np.ones(len(syn_scores))])
        scores = np.concatenate([good_scores, syn_scores])
        image_auc = safe_roc_auc(labels, scores)
        pixel_auc = sampled_pixel_auroc(
            [*good_maps, *syn_maps],
            [*good_zero_masks, *syn_masks],
            max_pixels=min(500_000, CFG["pixel_max_samples"]),
            seed=SEED,
        )
        objective = 0.62 * np.nan_to_num(image_auc) + 0.38 * np.nan_to_num(pixel_auc)
        weight_rows.append(
            {
                "category": category,
                "ensemble_weight_patchcore": float(weight),
                "image_auroc_synthetic": image_auc,
                "pixel_auroc_synthetic": pixel_auc,
                "objective": float(objective),
            }
        )

    weight_frame = pd.DataFrame(weight_rows).sort_values("objective", ascending=False).reset_index(drop=True)
    best_weight = float(weight_frame.iloc[0].ensemble_weight_patchcore)
    ENSEMBLE_HPO_ROWS.extend(weight_frame.to_dict("records"))
    print(f"{category}: selected PatchCore ensemble weight = {best_weight:.2f}")

    good_scores = best_weight * patch_good_scores + (1.0 - best_weight) * vae_good_scores
    syn_scores = best_weight * patch_syn_scores + (1.0 - best_weight) * vae_syn_scores
    good_maps = best_weight * patch_good_maps + (1.0 - best_weight) * vae_good_maps
    syn_maps = best_weight * patch_syn_maps + (1.0 - best_weight) * vae_syn_maps
    calibration_scores = np.concatenate([good_scores, syn_scores])
    calibration_labels = np.concatenate(
        [np.zeros(len(good_scores), dtype=np.int16), np.ones(len(syn_scores), dtype=np.int16)]
    )
    synthetic_threshold, threshold_curve = choose_cost_threshold(
        calibration_scores,
        calibration_labels,
        defect_prevalence=DEFECT_PREVALENCE,
        cost_escape=COST_ESCAPE,
        cost_false_reject=COST_FALSE_REJECT,
        check_band=CHECK_BAND,
        cost_review=COST_HUMAN_REVIEW,
    )

    patch_test_scores = normalize_with_scaler(patch_cache["test_scores_raw"], patch_cal["score_scaler"])
    vae_test_scores = normalize_with_scaler(vae_cache["test_scores_raw"], vae_cal["score_scaler"])
    patch_test_maps = normalize_with_scaler(patch_cache["test_maps"].astype(np.float32), patch_cal["pixel_scaler"])
    vae_test_maps = normalize_with_scaler(vae_cache["test_maps"].astype(np.float32), vae_cal["pixel_scaler"])
    test_scores = best_weight * patch_test_scores + (1.0 - best_weight) * vae_test_scores
    test_maps = best_weight * patch_test_maps + (1.0 - best_weight) * vae_test_maps

    ensemble_folder = ARTIFACT_DIR / category / "ensemble"
    ensemble_folder.mkdir(parents=True, exist_ok=True)
    threshold_curve.rename(columns={"threshold": "threshold_normalized"}).to_csv(
        ensemble_folder / "threshold_curve.csv", index=False
    )
    calibration = {
        "threshold_source": "held-out good plus synthetic anomalies; no real defects",
        "default_threshold_normalized": float(synthetic_threshold),
        "synthetic_threshold_normalized": float(synthetic_threshold),
        "check_band": CHECK_BAND,
        "score_scaler": {"low": 0.0, "high": 100.0, "lower_q": 0.0, "upper_q": 1.0},
        "pixel_scaler": {"low": 0.0, "high": 100.0, "lower_q": 0.0, "upper_q": 1.0},
        "output_size": int(CFG["patch_image_size"]),
        "ensemble_weight_patchcore": best_weight,
        "calibration_good_count": len(good_scores),
        "synthetic_count": len(syn_scores),
    }
    save_json(calibration, ensemble_folder / "calibration.json")
    weight_frame.to_csv(ensemble_folder / "hpo.csv", index=False)

    test_frame = pd.DataFrame(
        {
            "image_path": patch_cache["test_paths"].astype(str),
            "label": patch_cache["test_labels"].astype(int),
            "defect_type": patch_cache["test_defect_types"].astype(str),
            "mask_path": patch_cache["test_mask_paths"].astype(str),
        }
    )
    syn_frame = pd.DataFrame(
        {
            "image_path": patch_cache["syn_paths"].astype(str),
            "mask_path": patch_cache["syn_mask_paths"].astype(str),
        }
    )
    save_method_cache(
        ENSEMBLE_CACHE_DIR / f"{category}.npz",
        test_frame,
        test_scores,
        test_maps,
        patch_cache["cal_good_paths"].astype(str).tolist(),
        good_scores,
        good_maps,
        syn_frame,
        syn_scores,
        syn_maps,
    )
    copy_checkpoint(ensemble_folder, Path("artifacts") / category / "ensemble")
    copy_checkpoint(ENSEMBLE_CACHE_DIR / f"{category}.npz", Path("checkpoints") / "ensemble" / f"{category}.npz")

ensemble_hpo_all = pd.DataFrame(ENSEMBLE_HPO_ROWS)
ensemble_hpo_all.to_csv(REPORT_DIR / "ensemble_hpo.csv", index=False)

# ----------------------------- Formal evaluation -----------------------------
metrics_rows: list[dict] = []
operating_rows: list[dict] = []
threshold_rows: list[dict] = []
per_defect_rows: list[dict] = []
methods = ["patchcore", "vae", "mean", "ensemble"]

role_by_path = split_manifest.set_index("image_path")["role"].to_dict()


def qualitative_panel(category: str, method: str, cache, normalized_scores: np.ndarray, calibration: dict) -> None:
    labels = cache["test_labels"].astype(int)
    paths = cache["test_paths"].astype(str)
    defect_indices = np.where(labels == 1)[0]
    good_indices = np.where(labels == 0)[0]
    selected = []
    if len(defect_indices):
        selected.append((int(defect_indices[np.argmax(normalized_scores[defect_indices])]), "high-scoring defect"))
        selected.append((int(defect_indices[np.argmin(normalized_scores[defect_indices])]), "lowest-scoring defect / escape risk"))
    if len(good_indices):
        selected.append((int(good_indices[np.argmax(normalized_scores[good_indices])]), "highest-scoring good / false-reject risk"))
        selected.append((int(good_indices[np.argmin(normalized_scores[good_indices])]), "lowest-scoring good"))
    if not selected:
        return

    fig, axes = plt.subplots(len(selected), 3, figsize=(11, 3.2 * len(selected)), squeeze=False)
    maps = cache["test_maps"].astype(np.float32)
    mask_paths = cache["test_mask_paths"].astype(str)
    for row_index, (sample_index, description) in enumerate(selected):
        image = resize_rgb(read_rgb(paths[sample_index]), int(CFG["patch_image_size"]))
        normalized_map = normalize_with_scaler(maps[sample_index], calibration["pixel_scaler"])
        heat_bgr = cv2.applyColorMap(normalized_map.astype(np.uint8), cv2.COLORMAP_TURBO)
        heat_rgb = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(image, 0.55, heat_rgb, 0.45, 0)
        mask = read_mask(mask_paths[sample_index] or None, image_size=int(CFG["patch_image_size"]))
        axes[row_index, 0].imshow(image)
        axes[row_index, 0].set_title(f"{description}\nscore={normalized_scores[sample_index]:.1f}")
        axes[row_index, 1].imshow(overlay)
        axes[row_index, 1].set_title("Anomaly overlay")
        axes[row_index, 2].imshow(mask, cmap="gray")
        axes[row_index, 2].set_title("Ground-truth mask")
        for axis in axes[row_index]:
            axis.axis("off")
    fig.suptitle(f"{category.title()} — {method}")
    fig.tight_layout()
    save_and_show_figure(fig, f"qualitative_{category}_{method}.png")


for category in CATEGORIES:
    for method in methods:
        cache = load_method_cache(category, method)
        calibration_path = ARTIFACT_DIR / category / method / "calibration.json"
        calibration = json.loads(calibration_path.read_text(encoding="utf-8"))
        labels = cache["test_labels"].astype(int)
        raw_scores = cache["test_scores_raw"].astype(np.float32)
        normalized_scores = normalize_with_scaler(raw_scores, calibration["score_scaler"])
        maps = cache["test_maps"].astype(np.float32)
        mask_paths = cache["test_mask_paths"].astype(str)
        masks = [read_mask(path or None, image_size=maps.shape[-1]) for path in mask_paths]

        ranking = image_metrics(labels, normalized_scores)
        ci_low, ci_high = bootstrap_auc_ci(
            labels,
            normalized_scores,
            n_bootstrap=int(CFG["bootstrap_repeats"]),
            seed=SEED,
        )
        pixel_auc = sampled_pixel_auroc(
            maps,
            masks,
            max_pixels=int(CFG["pixel_max_samples"]),
            seed=SEED,
        )
        aupro_30 = compute_aupro(
            [np.asarray(anomaly_map, dtype=np.float32) for anomaly_map in maps],
            [np.asarray(mask, dtype=np.uint8) for mask in masks],
            fpr_limit=0.30,
            num_thresholds=int(CFG["aupro_thresholds"]),
        )

        metrics_rows.append(
            {
                "category": category,
                "method": method,
                **ranking,
                "image_auroc_ci_low": ci_low,
                "image_auroc_ci_high": ci_high,
                "pixel_auroc": pixel_auc,
                "aupro_30": aupro_30,
                "test_images": len(labels),
                "defective_images": int(labels.sum()),
            }
        )

        paths = cache["test_paths"].astype(str)
        roles = np.asarray([role_by_path.get(path, "unknown") for path in paths])
        oracle_indices = np.where(roles == "oracle_threshold_calibration")[0]
        operating_indices = np.where(roles == "operating_test")[0]
        synthetic_threshold = float(calibration["synthetic_threshold_normalized"])
        if len(oracle_indices) and len(np.unique(labels[oracle_indices])) == 2:
            real_threshold, _ = choose_cost_threshold(
                normalized_scores[oracle_indices],
                labels[oracle_indices],
                defect_prevalence=DEFECT_PREVALENCE,
                cost_escape=COST_ESCAPE,
                cost_false_reject=COST_FALSE_REJECT,
                check_band=CHECK_BAND,
                cost_review=COST_HUMAN_REVIEW,
            )
        else:
            real_threshold = float("nan")

        synthetic_operating = three_way_operating_metrics(
            normalized_scores[operating_indices],
            labels[operating_indices],
            threshold=synthetic_threshold,
            check_band=CHECK_BAND,
            defect_prevalence=DEFECT_PREVALENCE,
            cost_escape=COST_ESCAPE,
            cost_false_reject=COST_FALSE_REJECT,
            cost_review=COST_HUMAN_REVIEW,
        )
        operating_rows.append(
            {
                "category": category,
                "method": method,
                "threshold_source": "synthetic_only",
                **synthetic_operating,
                "evaluation_images": len(operating_indices),
            }
        )

        if np.isfinite(real_threshold):
            real_operating = three_way_operating_metrics(
                normalized_scores[operating_indices],
                labels[operating_indices],
                threshold=float(real_threshold),
                check_band=CHECK_BAND,
                defect_prevalence=DEFECT_PREVALENCE,
                cost_escape=COST_ESCAPE,
                cost_false_reject=COST_FALSE_REJECT,
                cost_review=COST_HUMAN_REVIEW,
            )
            operating_rows.append(
                {
                    "category": category,
                    "method": method,
                    "threshold_source": "real_label_oracle",
                    **real_operating,
                    "evaluation_images": len(operating_indices),
                }
            )
            gap = threshold_distance(synthetic_threshold, float(real_threshold), score_scale=100.0)
        else:
            real_operating = {}
            gap = {"threshold_absolute_gap": float("nan"), "threshold_scale_normalized_gap": float("nan")}

        threshold_rows.append(
            {
                "category": category,
                "method": method,
                "synthetic_threshold_normalized": synthetic_threshold,
                "real_threshold_normalized": real_threshold,
                **gap,
                "oracle_calibration_images": len(oracle_indices),
                "operating_test_images": len(operating_indices),
            }
        )

        for defect_type in sorted(set(cache["test_defect_types"].astype(str)) - {"good"}):
            indices = np.where((cache["test_defect_types"].astype(str) == defect_type) & (roles == "operating_test"))[0]
            if len(indices) == 0:
                continue
            decisions = [decision_from_score(float(normalized_scores[i]), synthetic_threshold, CHECK_BAND) for i in indices]
            per_defect_rows.append(
                {
                    "category": category,
                    "method": method,
                    "defect_type": defect_type,
                    "images": len(indices),
                    "escape_rate": float(np.mean(np.asarray(decisions) == "PASS")),
                    "check_rate": float(np.mean(np.asarray(decisions) == "CHECK")),
                    "reject_rate": float(np.mean(np.asarray(decisions) == "REJECT")),
                    "mean_score": float(np.mean(normalized_scores[indices])),
                }
            )

        calibration.update(
            {
                "image_auroc": ranking["image_auroc"],
                "image_average_precision": ranking["image_average_precision"],
                "image_auroc_ci_low": ci_low,
                "image_auroc_ci_high": ci_high,
                "pixel_auroc": pixel_auc,
                "aupro_30": aupro_30,
                "real_threshold_normalized": float(real_threshold) if np.isfinite(real_threshold) else None,
                "false_reject_rate": synthetic_operating["false_reject_rate"],
                "escape_rate": synthetic_operating["escape_rate"],
                "review_rate": synthetic_operating["review_rate"],
                "expected_cost_per_1000": synthetic_operating["expected_cost_per_1000"],
            }
        )
        save_json(calibration, calibration_path)
        copy_checkpoint(ARTIFACT_DIR / category / method, Path("artifacts") / category / method)
        qualitative_panel(category, method, cache, normalized_scores, calibration)

metrics_df = pd.DataFrame(metrics_rows)
operating_df = pd.DataFrame(operating_rows)
threshold_df = pd.DataFrame(threshold_rows)
per_defect_df = pd.DataFrame(per_defect_rows)
cost_model_df = pd.DataFrame(
    [
        {"parameter": "defect_prevalence", "value": DEFECT_PREVALENCE, "unit": "fraction of production parts", "meaning": "Expected long-run defective-part rate"},
        {"parameter": "cost_escape", "value": COST_ESCAPE, "unit": "relative cost per escaped defect", "meaning": "Downstream cost of releasing a defective part"},
        {"parameter": "cost_false_reject", "value": COST_FALSE_REJECT, "unit": "relative cost per good rejected part", "meaning": "Scrap, rework, and disruption cost"},
        {"parameter": "cost_human_review", "value": COST_HUMAN_REVIEW, "unit": "relative cost per CHECK", "meaning": "Inspector time and handling cost"},
        {"parameter": "check_band", "value": CHECK_BAND, "unit": "normalized score points on each side", "meaning": "Width of the human-review region"},
    ]
)

metric_summary_columns = [
    "image_auroc",
    "image_average_precision",
    "pixel_auroc",
    "aupro_30",
]
metrics_summary_df = (
    metrics_df.groupby("method", as_index=False)[metric_summary_columns]
    .mean(numeric_only=True)
    .rename(columns={column: f"macro_mean_{column}" for column in metric_summary_columns})
)
operating_summary_df = (
    operating_df[operating_df.threshold_source == "synthetic_only"]
    .groupby("method", as_index=False)[
        [
            "false_reject_rate",
            "escape_rate",
            "review_rate",
            "false_reject_cost_per_1000",
            "escape_cost_per_1000",
            "review_cost_per_1000",
            "expected_cost_per_1000",
        ]
    ]
    .mean(numeric_only=True)
)

metrics_df.to_csv(REPORT_DIR / "metrics_by_category_method.csv", index=False)
operating_df.to_csv(REPORT_DIR / "operating_metrics.csv", index=False)
metrics_summary_df.to_csv(REPORT_DIR / "metrics_macro_summary.csv", index=False)
operating_summary_df.to_csv(REPORT_DIR / "operating_macro_summary.csv", index=False)
threshold_df.to_csv(REPORT_DIR / "threshold_comparison.csv", index=False)
per_defect_df.to_csv(REPORT_DIR / "per_defect_type_metrics.csv", index=False)
cost_model_df.to_csv(REPORT_DIR / "cost_model.csv", index=False)
ensemble_hpo_all.to_csv(REPORT_DIR / "ensemble_hpo.csv", index=False)

for report_name in [
    "metrics_by_category_method.csv",
    "operating_metrics.csv",
    "metrics_macro_summary.csv",
    "operating_macro_summary.csv",
    "threshold_comparison.csv",
    "per_defect_type_metrics.csv",
    "cost_model.csv",
    "ensemble_hpo.csv",
]:
    copy_checkpoint(REPORT_DIR / report_name, Path("reports") / report_name)

print("\nRanking and localization metrics")
print(metrics_df.round(4).to_string(index=False))
print("\nOperating metrics")
print(operating_df.round(4).to_string(index=False))
print("\nThreshold comparison")
print(threshold_df.round(4).to_string(index=False))

fig, axis = plt.subplots(figsize=(11, 5))
metrics_df.pivot(index="category", columns="method", values="image_auroc").plot(kind="bar", ax=axis)
axis.set_ylim(0, 1.02)
axis.set_ylabel("Image AUROC")
axis.set_title("Image-level anomaly detection by category and method")
axis.tick_params(axis="x", rotation=0)
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_and_show_figure(fig, "results_image_auroc.png")

fig, axis = plt.subplots(figsize=(11, 5))
metrics_df.pivot(index="category", columns="method", values="pixel_auroc").plot(kind="bar", ax=axis)
axis.set_ylim(0, 1.02)
axis.set_ylabel("Pixel AUROC")
axis.set_title("Pixel-level localization by category and method")
axis.tick_params(axis="x", rotation=0)
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_and_show_figure(fig, "results_pixel_auroc.png")

synthetic_operating_plot = operating_df[operating_df.threshold_source == "synthetic_only"]
fig, axis = plt.subplots(figsize=(11, 5))
synthetic_operating_plot.pivot(index="category", columns="method", values="expected_cost_per_1000").plot(kind="bar", ax=axis)
axis.set_ylabel("Expected cost per 1,000 parts")
axis.set_title("Cost-sensitive operating comparison at the synthetic-only threshold")
axis.tick_params(axis="x", rotation=0)
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_and_show_figure(fig, "results_expected_cost.png")

# Cost breakdown makes the business trade-off auditable instead of hiding it in one total.
cost_breakdown = synthetic_operating_plot.copy()
cost_breakdown["category_method"] = cost_breakdown["category"] + " · " + cost_breakdown["method"]
fig, axis = plt.subplots(figsize=(13, 6))
cost_breakdown.set_index("category_method")[[
    "false_reject_cost_per_1000",
    "escape_cost_per_1000",
    "review_cost_per_1000",
]].plot(kind="bar", stacked=True, ax=axis)
axis.set_ylabel("Expected cost per 1,000 parts")
axis.set_xlabel("")
axis.set_title("Expected-cost components at the synthetic-only threshold")
axis.tick_params(axis="x", rotation=55)
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_and_show_figure(fig, "results_cost_breakdown.png")

operating_rates = synthetic_operating_plot.copy()
operating_rates["category_method"] = operating_rates["category"] + " · " + operating_rates["method"]
fig, axis = plt.subplots(figsize=(13, 6))
operating_rates.set_index("category_method")[["false_reject_rate", "escape_rate", "review_rate"]].plot(
    kind="bar", ax=axis
)
axis.set_ylim(0, 1.02)
axis.set_ylabel("Rate")
axis.set_xlabel("")
axis.set_title("False rejects, escapes, and prevalence-adjusted review load")
axis.tick_params(axis="x", rotation=55)
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_and_show_figure(fig, "results_operating_rates.png")

fig, axis = plt.subplots(figsize=(12, 5))
threshold_plot = threshold_df.copy()
threshold_plot["category_method"] = threshold_plot["category"] + " · " + threshold_plot["method"]
threshold_plot.set_index("category_method")[["threshold_absolute_gap"]].plot(kind="bar", legend=False, ax=axis)
axis.set_ylabel("Absolute gap on 0–100 score scale")
axis.set_xlabel("")
axis.set_title("Synthetic-only threshold versus real-label oracle")
axis.tick_params(axis="x", rotation=55)
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_and_show_figure(fig, "results_threshold_gap.png")

mark_stage(
    "evaluation_complete",
    metric_rows=len(metrics_df),
    operating_rows=len(operating_df),
    threshold_rows=len(threshold_df),
)


## 18 · Measure latency and artifact footprint

After warm-up, inference is timed repeatedly through the same packaged runtime used by the UI. Median, mean, and p95 latency are saved. Every artifact file is measured so the ONNX graph, memory banks, VAEs, and baselines can be checked against deployment constraints.

In [ ]:
from __future__ import annotations

from defectspotter.runtime import DefectSpotterRuntime

onnx_required = ARTIFACT_DIR / "onnx" / "mobilenet_v3_small_patch.onnx"
if not onnx_required.exists():
    raise FileNotFoundError(f"Run the ONNX export cell first; missing: {onnx_required}")
runtime = DefectSpotterRuntime(ARTIFACT_DIR, device=DEVICE)
latency_rows = []

for category in CATEGORIES:
    sample_paths = selected_test_frame(category).image_path.tolist()
    if not sample_paths:
        continue
    sample_paths = sample_paths[: min(5, len(sample_paths))]
    for method in runtime.methods(category):
        # Warm up model loading and caches before timing.
        for path in sample_paths[:2]:
            runtime.inspect(path, category, method)
        durations = []
        for repeat in range(int(CFG["latency_repeats"])):
            path = sample_paths[repeat % len(sample_paths)]
            started = time.perf_counter()
            runtime.inspect(path, category, method)
            durations.append((time.perf_counter() - started) * 1000.0)
        row = {
            "category": category,
            "method": method,
            "device": DEVICE,
            "latency_ms_median": float(np.median(durations)),
            "latency_ms_p95": float(np.quantile(durations, 0.95)),
            "latency_ms_mean": float(np.mean(durations)),
            "repeats": len(durations),
        }
        latency_rows.append(row)
        calibration_path = ARTIFACT_DIR / category / method / "calibration.json"
        calibration = json.loads(calibration_path.read_text(encoding="utf-8"))
        calibration.update({key: value for key, value in row.items() if key.startswith("latency_")})
        save_json(calibration, calibration_path)
        copy_checkpoint(ARTIFACT_DIR / category / method, Path("artifacts") / category / method)

latency_df = pd.DataFrame(latency_rows)
latency_df.to_csv(REPORT_DIR / "latency.csv", index=False)
copy_checkpoint(REPORT_DIR / "latency.csv", "reports/latency.csv")
print(latency_df.round(3).to_string(index=False))

artifact_size_rows = []
for path in sorted(ARTIFACT_DIR.rglob("*")):
    if path.is_file():
        relative = path.relative_to(ARTIFACT_DIR)
        parts = relative.parts
        category = parts[0] if parts and parts[0] in CATEGORIES else "shared"
        method = parts[1] if len(parts) > 1 and parts[0] in CATEGORIES else parts[0]
        artifact_size_rows.append(
            {
                "category": category,
                "method_or_group": method,
                "file": str(relative),
                "size_mb": path.stat().st_size / 1024**2,
            }
        )
artifact_sizes_df = pd.DataFrame(artifact_size_rows)
artifact_sizes_df.to_csv(REPORT_DIR / "artifact_sizes.csv", index=False)
copy_checkpoint(REPORT_DIR / "artifact_sizes.csv", "reports/artifact_sizes.csv")

print("\nLargest packaged artifacts")
print(artifact_sizes_df.sort_values("size_mb", ascending=False).head(20).round(3).to_string(index=False))

fig, axis = plt.subplots(figsize=(11, 5))
latency_df.pivot(index="category", columns="method", values="latency_ms_median").plot(kind="bar", ax=axis)
axis.set_ylabel("Median latency (ms/part)")
axis.set_title(f"Inference latency on {DEVICE}")
axis.tick_params(axis="x", rotation=0)
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_and_show_figure(fig, "engineering_latency.png")

mark_stage("latency_and_size_complete", latency_rows=len(latency_df), artifact_files=len(artifact_sizes_df))


## 19 · Stress-test camera and lighting drift

Good calibration images are passed through a simulated stream with changing brightness, contrast, and blur. The notebook logs acquisition statistics, anomaly score, global feature drift, rolling statistics, PSI, and alert status. This is a monitoring demonstration, not a universal alert calibration.

In [ ]:
from __future__ import annotations

if "runtime" not in globals():
    from defectspotter.runtime import DefectSpotterRuntime
    runtime = DefectSpotterRuntime(ARTIFACT_DIR, device=DEVICE)


def array_acquisition_stats(image: np.ndarray) -> dict[str, float]:
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    return {
        "brightness": float(gray.mean()),
        "contrast": float(gray.std()),
        "sharpness": float(cv2.Laplacian(gray, cv2.CV_32F).var()),
    }


def perturb_image(image: np.ndarray, brightness: float, contrast: float, blur_radius: float) -> np.ndarray:
    pil = Image.fromarray(image)
    pil = ImageEnhance.Brightness(pil).enhance(brightness)
    pil = ImageEnhance.Contrast(pil).enhance(contrast)
    if blur_radius > 0:
        pil = pil.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    return np.asarray(pil.convert("RGB"))


drift_rows = []
for category in CATEGORIES:
    good_paths = split_manifest[
        (split_manifest.category == category) & (split_manifest.role == "calibration_good")
    ].image_path.tolist()
    if len(good_paths) > CFG["drift_samples"]:
        good_paths = good_paths[: int(CFG["drift_samples"])]

    reference_scores = []
    stream_records = []
    for index, path in enumerate(good_paths):
        image = read_rgb(path)
        result = runtime.inspect(image, category, "patchcore")
        reference_scores.append(result["normalized_score"])
        stream_records.append(
            {
                "phase": "reference_good",
                "brightness_factor": 1.0,
                "contrast_factor": 1.0,
                "blur_radius": 0.0,
                "normalized_score": result["normalized_score"],
                "global_feature_drift": result["auxiliary"].get("drift_score", np.nan),
                **array_acquisition_stats(image),
            }
        )

    # The simulated stream gradually departs from the qualified imaging conditions.
    for index, path in enumerate(good_paths):
        image = read_rgb(path)
        progress = index / max(1, len(good_paths) - 1)
        brightness_factor = 0.70 + 0.65 * progress
        contrast_factor = 0.78 + 0.48 * progress
        blur_radius = 0.0 if progress < 0.35 else 3.0 * (progress - 0.35) / 0.65
        changed = perturb_image(image, brightness_factor, contrast_factor, blur_radius)
        result = runtime.inspect(changed, category, "patchcore")
        stream_records.append(
            {
                "phase": "simulated_camera_drift",
                "brightness_factor": brightness_factor,
                "contrast_factor": contrast_factor,
                "blur_radius": blur_radius,
                "normalized_score": result["normalized_score"],
                "global_feature_drift": result["auxiliary"].get("drift_score", np.nan),
                **array_acquisition_stats(changed),
            }
        )

    score_stream = [row["normalized_score"] for row in stream_records]
    alerts = drift_alert_table(
        score_stream,
        reference_scores,
        window=max(5, min(12, len(reference_scores))),
        psi_warn=0.10,
        psi_alert=0.25,
    )
    for stream_index, (record, alert) in enumerate(zip(stream_records, alerts.to_dict("records"))):
        drift_rows.append(
            {
                "category": category,
                "stream_index": stream_index,
                **record,
                "rolling_mean": alert["rolling_mean"],
                "rolling_p95": alert["rolling_p95"],
                "psi": alert["psi"],
                "drift_status": alert["drift_status"],
            }
        )

drift_df = pd.DataFrame(drift_rows)
drift_df.to_csv(REPORT_DIR / "drift_stress_test.csv", index=False)
copy_checkpoint(REPORT_DIR / "drift_stress_test.csv", "reports/drift_stress_test.csv")
print(drift_df.groupby(["category", "phase", "drift_status"]).size().rename("samples").reset_index().to_string(index=False))

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for category, group in drift_df.groupby("category"):
    axes[0].plot(group.stream_index, group.normalized_score, label=category)
    axes[1].plot(group.stream_index, group.psi, label=category)
axes[0].set_ylabel("Normalized anomaly score")
axes[1].set_ylabel("PSI")
axes[1].set_xlabel("Stream index")
axes[0].set_title("Simulated lighting/focus drift response")
axes[0].grid(alpha=0.25)
axes[1].grid(alpha=0.25)
axes[0].legend()
axes[1].legend()
fig.tight_layout()
save_and_show_figure(fig, "drift_score_and_psi.png")

mark_stage("drift_stress_test_complete", rows=len(drift_df))


## 20 · Finalize documentation, examples, tests, ZIP, Drive copy, and browser download

The completed project is reloaded from disk, a small attributed MVTec research example set is added for the UI, run-specific evidence is written into the documentation, and the exact package is tested before archiving. The ZIP contains source, requirements, trained artifacts, reports, figures, examples, deployment templates, Gradio and Streamlit apps, and tests—while excluding the full raw dataset.


In [ ]:
from __future__ import annotations

import json
import subprocess
import re

# Build a small attributed research example set for the UI. Raw training data remain excluded.
EXAMPLES_DIR = PROJECT_DIR / "examples"
if EXAMPLES_DIR.exists():
    shutil.rmtree(EXAMPLES_DIR)
EXAMPLES_DIR.mkdir(parents=True, exist_ok=True)
(EXAMPLES_DIR / "masks").mkdir(parents=True, exist_ok=True)

index_manifest_path = MANIFEST_DIR / "dataset_index.csv"
if not index_manifest_path.exists():
    raise FileNotFoundError(f"Missing dataset index required for examples: {index_manifest_path}")
example_index = pd.read_csv(index_manifest_path)
example_rows = []
for category_name in CATEGORIES:
    category_test = example_index[(example_index.category == category_name) & (example_index.split == "test")].copy()
    selections = []
    good_rows = category_test[category_test.label.eq(0)]
    if not good_rows.empty:
        selections.append(good_rows.iloc[0])
    for _, defect_group in category_test[category_test.label.eq(1)].groupby("defect_type"):
        selections.append(defect_group.iloc[0])
    for row in selections:
        source_path = Path(row.image_path)
        safe_defect = re.sub(r"[^A-Za-z0-9_-]+", "_", str(row.defect_type))
        output_name = f"{category_name}_{safe_defect}_{source_path.stem}{source_path.suffix.lower()}"
        shutil.copy2(source_path, EXAMPLES_DIR / output_name)
        mask_name = ""
        if isinstance(row.mask_path, str) and row.mask_path.strip() and Path(row.mask_path).exists():
            mask_name = f"{category_name}_{safe_defect}_{Path(row.mask_path).name}"
            shutil.copy2(row.mask_path, EXAMPLES_DIR / "masks" / mask_name)
        example_rows.append({
            "file": output_name,
            "mask_file": mask_name,
            "category": category_name,
            "label": int(row.label),
            "defect_type": str(row.defect_type),
            "source_dataset": "MVTec AD",
            "source_mirror": HF_REPO_ID,
            "license": "CC BY-NC-SA 4.0; non-commercial",
        })

pd.DataFrame(example_rows).to_csv(EXAMPLES_DIR / "examples_manifest.csv", index=False)
(EXAMPLES_DIR / "ATTRIBUTION.md").write_text(
    "# Research example attribution\n\nThese images are selected from MVTec AD for research and teaching. "
    "MVTec AD is CC BY-NC-SA 4.0 and non-commercial. Original dataset: "
    "https://www.mvtec.com/company/research/datasets/mvtec-ad\n",
    encoding="utf-8",
)
copy_checkpoint(EXAMPLES_DIR, "examples")
print(f"Prepared {len(example_rows)} attributed UI examples in {EXAMPLES_DIR}")



def require_csv(filename: str) -> pd.DataFrame:
    path = REPORT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Required report is missing: {path}. Run the preceding analysis cell.")
    return pd.read_csv(path)


# Reload from disk when this cell is run after a partial restart or manual cell selection.
metrics_df = globals().get("metrics_df") if isinstance(globals().get("metrics_df"), pd.DataFrame) else require_csv("metrics_by_category_method.csv")
operating_df = globals().get("operating_df") if isinstance(globals().get("operating_df"), pd.DataFrame) else require_csv("operating_metrics.csv")
threshold_df = globals().get("threshold_df") if isinstance(globals().get("threshold_df"), pd.DataFrame) else require_csv("threshold_comparison.csv")
latency_df = globals().get("latency_df") if isinstance(globals().get("latency_df"), pd.DataFrame) else require_csv("latency.csv")
artifact_sizes_df = globals().get("artifact_sizes_df") if isinstance(globals().get("artifact_sizes_df"), pd.DataFrame) else require_csv("artifact_sizes.csv")

environment_path = PROJECT_DIR / "environment.json"
ENVIRONMENT = globals().get("ENVIRONMENT") or json.loads(environment_path.read_text(encoding="utf-8"))
onnx_report_path = ARTIFACT_DIR / "onnx" / "export_report.json"
if not onnx_report_path.exists():
    raise FileNotFoundError(f"Missing verified ONNX report: {onnx_report_path}")
onnx_result = json.loads(onnx_report_path.read_text(encoding="utf-8"))
if not onnx_result.get("verified", False):
    raise RuntimeError(f"ONNX report is not verified: {onnx_result}")

metrics_preview = metrics_df.round(4).to_string(index=False)
operating_preview = operating_df[operating_df.threshold_source == "synthetic_only"].round(4).to_string(index=False)
latency_preview = latency_df.round(3).to_string(index=False)

readme_path = PROJECT_DIR / "README.md"
readme_text = readme_path.read_text(encoding="utf-8")
run_marker = f"## Completed run: `{RUN_ID}`"
run_appendix = f"""

## Completed run: `{RUN_ID}`

Profile: `{PROFILE}`
Categories: `{', '.join(CATEGORIES)}`
Compute device during packaging: `{DEVICE}`

### Ranking and localization metrics

```text
{metrics_preview}
```

### Operating metrics at the synthetic-only threshold

```text
{operating_preview}
```

### Measured latency

```text
{latency_preview}
```

The full machine-readable tables are in `reports/`. The real-label threshold is included only as an oracle comparison; the packaged default remains synthetic-only.
"""
if run_marker not in readme_text:
    readme_text += run_appendix
readme_path.write_text(readme_text, encoding="utf-8")

model_card_path = PROJECT_DIR / "MODEL_CARD.md"
model_card_text = model_card_path.read_text(encoding="utf-8")
model_run_marker = f"## Run-specific evidence\n\nRun ID: `{RUN_ID}`"
model_run_appendix = f"""

## Run-specific evidence

Run ID: `{RUN_ID}`

```text
{metrics_preview}
```

See `reports/operating_metrics.csv`, `reports/threshold_comparison.csv`, `reports/latency.csv`, and each method's `calibration.json` for the complete operating record.
"""
if model_run_marker not in model_card_text:
    model_card_text += model_run_appendix
model_card_path.write_text(model_card_text, encoding="utf-8")

run_summary = {
    "run_id": RUN_ID,
    "profile": PROFILE,
    "categories": CATEGORIES,
    "environment": ENVIRONMENT,
    "metrics_records": metrics_df.to_dict("records"),
    "operating_records": operating_df.to_dict("records"),
    "threshold_records": threshold_df.to_dict("records"),
    "latency_records": latency_df.to_dict("records"),
    "artifact_size_records": artifact_sizes_df.to_dict("records"),
    "onnx_export": onnx_result,
    "dataset_license": "MVTec AD: CC BY-NC-SA 4.0, non-commercial",
    "huggingface_space_repo": f"{HF_USERNAME}/{HF_SPACE_NAME}",
    "huggingface_model_repo": f"{HF_USERNAME}/{HF_MODEL_REPO_NAME}",
}
save_json(run_summary, REPORT_DIR / "run_summary.json")
copy_checkpoint(REPORT_DIR, "reports")
copy_checkpoint(FIGURE_DIR, "figures")
copy_checkpoint(MANIFEST_DIR, "manifests")
copy_checkpoint(EXAMPLES_DIR, "examples")
copy_checkpoint(ARTIFACT_DIR, "artifacts")

# Validate the exact project that will be archived.
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_DIR,
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.returncode != 0:
    print(test_result.stderr)
    raise RuntimeError("Local package tests failed; ZIP was not created.")

for cache_dir in PROJECT_DIR.rglob("__pycache__"):
    shutil.rmtree(cache_dir, ignore_errors=True)
for cache_name in [".pytest_cache", ".ipynb_checkpoints"]:
    for cache_dir in PROJECT_DIR.rglob(cache_name):
        shutil.rmtree(cache_dir, ignore_errors=True)
for pyc in PROJECT_DIR.rglob("*.pyc"):
    pyc.unlink(missing_ok=True)

zip_base = SESSION_ROOT / f"DefectSpotter_{RUN_ID}"
zip_path = Path(shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=PROJECT_DIR.parent,
    base_dir=PROJECT_DIR.name,
))
drive_zip = DRIVE_RUN_DIR / zip_path.name
shutil.copy2(zip_path, drive_zip)

print(f"Project ZIP created: {zip_path} ({zip_path.stat().st_size / 1024**2:.2f} MB)")
print(f"Project ZIP stored in the persistent run folder: {drive_zip}")
print("Top-level project contents:")
for path in sorted(PROJECT_DIR.iterdir()):
    print(f" - {path.name}")

mark_stage("package_complete", zip_path=str(drive_zip), zip_size_mb=zip_path.stat().st_size / 1024**2, tests="passed")

if AUTO_DOWNLOAD_ZIP and IN_COLAB and colab_files is not None:
    print("Starting the browser download. The same ZIP is already stored in the persistent run folder.")
    colab_files.download(str(zip_path))
else:
    print(f"ZIP available at: {zip_path}")


## 21 · Publish the backend artifacts and frontend to Hugging Face

This cell creates or updates two repositories under **`bhkr0901`**:

- `bhkr0901/defectspotter-artifacts` — trained models, reports, figures, examples, and run metadata;
- `bhkr0901/defectspotter` — a Gradio Space with interactive EDA, charts, upload/webcam/clipboard inspection, live streaming, batch analysis, anomaly maps, drift monitoring, and API endpoints.

Create a Hugging Face access token with write permission and store it in **Colab → Secrets** as `HF_TOKEN`. The token is read privately and is never saved in the notebook, ZIP, Drive output, or Hub repository.


In [ ]:
from __future__ import annotations

import getpass
import os
import shutil
import time
from pathlib import Path
from IPython.display import HTML, display
from huggingface_hub import HfApi

MODEL_REPO_ID = f"{HF_USERNAME}/{HF_MODEL_REPO_NAME}"
SPACE_REPO_ID = f"{HF_USERNAME}/{HF_SPACE_NAME}"
HF_DEPLOY_ROOT = SESSION_ROOT / "huggingface_deploy" / RUN_ID
MODEL_BUNDLE_DIR = HF_DEPLOY_ROOT / "model_repo"
SPACE_BUNDLE_DIR = HF_DEPLOY_ROOT / "space_repo"


def copy_path(source: Path, destination: Path) -> None:
    if source.is_dir():
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source, destination)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)


def build_hub_bundles() -> None:
    if HF_DEPLOY_ROOT.exists():
        shutil.rmtree(HF_DEPLOY_ROOT)
    MODEL_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
    SPACE_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

    # Backend evidence/model repository.
    for name in ["artifacts", "reports", "figures", "examples", "manifests"]:
        source = PROJECT_DIR / name
        if source.exists():
            copy_path(source, MODEL_BUNDLE_DIR / name)
    for name in ["run_config.json", "environment.json", "DATA_CARD.md", "MODEL_CARD.md", "LICENSE_NOTES.md"]:
        source = PROJECT_DIR / name
        if source.exists():
            copy_path(source, MODEL_BUNDLE_DIR / name)
    model_readme = (PROJECT_DIR / "MODEL_REPO_README_TEMPLATE.md").read_text(encoding="utf-8")
    model_readme = model_readme.replace("__SPACE_REPO_ID__", SPACE_REPO_ID)
    (MODEL_BUNDLE_DIR / "README.md").write_text(model_readme, encoding="utf-8")

    # Frontend and inference API Space repository.
    for name in ["app.py", "gradio_app.py", "LICENSE", "LICENSE_NOTES.md", "HUGGINGFACE_DEPLOYMENT.md"]:
        copy_path(PROJECT_DIR / name, SPACE_BUNDLE_DIR / name)
    copy_path(PROJECT_DIR / "defectspotter", SPACE_BUNDLE_DIR / "defectspotter")
    shutil.copy2(PROJECT_DIR / "requirements-space.txt", SPACE_BUNDLE_DIR / "requirements.txt")
    space_readme = (PROJECT_DIR / "SPACE_README_TEMPLATE.md").read_text(encoding="utf-8")
    space_readme = space_readme.replace("__MODEL_REPO_ID__", MODEL_REPO_ID)
    if HF_REPOSITORIES_PRIVATE:
        # Hub build-time preloading supports public repos only; private artifacts load at runtime via HF_TOKEN.
        space_readme = space_readme.replace(
            f"preload_from_hub:\n  - {MODEL_REPO_ID}\n", ""
        )
    (SPACE_BUNDLE_DIR / "README.md").write_text(space_readme, encoding="utf-8")

    # A local manifest makes the deployment auditable.
    bundle_manifest = {
        "run_id": RUN_ID,
        "model_repo_id": MODEL_REPO_ID,
        "space_repo_id": SPACE_REPO_ID,
        "model_files": sorted(p.relative_to(MODEL_BUNDLE_DIR).as_posix() for p in MODEL_BUNDLE_DIR.rglob("*") if p.is_file()),
        "space_files": sorted(p.relative_to(SPACE_BUNDLE_DIR).as_posix() for p in SPACE_BUNDLE_DIR.rglob("*") if p.is_file()),
    }
    atomic_write_json(bundle_manifest, HF_DEPLOY_ROOT / "bundle_manifest.json")
    copy_checkpoint(HF_DEPLOY_ROOT, "huggingface_deploy")


build_hub_bundles()
print(f"Model bundle: {MODEL_BUNDLE_DIR}")
print(f"Space bundle: {SPACE_BUNDLE_DIR}")

if not DEPLOY_TO_HUGGINGFACE:
    print("DEPLOY_TO_HUGGINGFACE is False. Bundles were built but not uploaded.")
else:
    token = os.environ.get("HF_TOKEN", "").strip()
    if not token and IN_COLAB:
        try:
            from google.colab import userdata
            token = (userdata.get("HF_TOKEN") or "").strip()
        except Exception:
            token = ""
    if not token:
        token = getpass.getpass("Enter a Hugging Face WRITE token (input is hidden): ").strip()
    if not token:
        raise RuntimeError("No Hugging Face token was supplied. Add the Colab secret HF_TOKEN and rerun only this cell.")

    api = HfApi(token=token)
    identity = api.whoami(token=token)
    allowed_names = {str(identity.get("name", ""))}
    for organization in identity.get("orgs", []) or []:
        if isinstance(organization, dict):
            allowed_names.add(str(organization.get("name", "")))
        else:
            allowed_names.add(str(organization))
    if HF_USERNAME not in allowed_names:
        raise PermissionError(
            f"The token belongs to {identity.get('name')!r} and does not show write access to {HF_USERNAME!r}. "
            "Use the correct account or organization token."
        )

    print(f"Authenticated as {identity.get('name')}. Uploading backend artifacts...")
    api.create_repo(
        repo_id=MODEL_REPO_ID,
        repo_type="model",
        private=HF_REPOSITORIES_PRIVATE,
        exist_ok=True,
        token=token,
    )
    model_commit = api.upload_folder(
        folder_path=str(MODEL_BUNDLE_DIR),
        repo_id=MODEL_REPO_ID,
        repo_type="model",
        token=token,
        commit_message=f"DefectSpotter artifacts from run {RUN_ID}",
        ignore_patterns=["**/__pycache__/**", "**/*.pyc", "**/.pytest_cache/**"],
    )

    print("Uploading the Gradio frontend and inference API...")
    space_creation_errors = []
    requested_hardware = HF_SPACE_HARDWARE or None
    hardware_attempts = [requested_hardware]
    if requested_hardware is not None:
        hardware_attempts.append(None)
    space_repo_created = False
    selected_hardware = None
    for hardware in hardware_attempts:
        try:
            create_kwargs = dict(
                repo_id=SPACE_REPO_ID,
                repo_type="space",
                space_sdk="gradio",
                private=HF_REPOSITORIES_PRIVATE,
                exist_ok=True,
                token=token,
            )
            if hardware is not None:
                create_kwargs["space_hardware"] = hardware
            api.create_repo(**create_kwargs)
            selected_hardware = hardware or "account-default"
            space_repo_created = True
            print(f"Space repository ready with hardware request: {selected_hardware}")
            break
        except Exception as exc:
            space_creation_errors.append(f"{hardware or 'account-default'}: {type(exc).__name__}: {exc}")
            print(f"Space creation attempt failed for {hardware or 'account-default'}: {exc}")
    if not space_repo_created:
        error_text = "\n".join(space_creation_errors)
        raise RuntimeError(
            "Hugging Face rejected creation of the compute Space. Eligible personal accounts can use "
            "ZeroGPU; otherwise the account needs a qualifying plan or billing setup. The model and "
            "Space bundles are already saved in Drive, so training does not need to be repeated.\n" + error_text
        )
    api.add_space_variable(
        repo_id=SPACE_REPO_ID,
        key="MODEL_REPO_ID",
        value=MODEL_REPO_ID,
        description="DefectSpotter trained artifact repository",
        token=token,
    )
    if HF_REPOSITORIES_PRIVATE:
        print("Private artifact mode: storing the current token as a Space secret for artifact download.")
        api.add_space_secret(
            repo_id=SPACE_REPO_ID,
            key="HF_TOKEN",
            value=token,
            description="Read access to the private DefectSpotter artifact repository",
            token=token,
        )
    space_commit = api.upload_folder(
        folder_path=str(SPACE_BUNDLE_DIR),
        repo_id=SPACE_REPO_ID,
        repo_type="space",
        token=token,
        commit_message=f"Deploy DefectSpotter UI/API from run {RUN_ID}",
        ignore_patterns=["**/__pycache__/**", "**/*.pyc", "**/.pytest_cache/**"],
    )

    stage_history = []
    final_stage = "UPLOAD_COMPLETE"
    if HF_WAIT_FOR_BUILD:
        deadline = time.time() + int(HF_BUILD_TIMEOUT_SECONDS)
        last_stage = None
        while time.time() < deadline:
            runtime_state = api.get_space_runtime(repo_id=SPACE_REPO_ID, token=token)
            stage = str(runtime_state.stage)
            if stage != last_stage:
                print(f"Space stage: {stage}")
                stage_history.append(stage)
                last_stage = stage
            final_stage = stage
            if stage == "RUNNING" or "ERROR" in stage or "PAUSED" in stage:
                break
            time.sleep(10)

    space_info = api.space_info(repo_id=SPACE_REPO_ID, token=token)
    hub_url = f"https://huggingface.co/spaces/{SPACE_REPO_ID}"
    direct_url = None
    host = getattr(space_info, "host", None)
    if host:
        direct_url = host if str(host).startswith("http") else f"https://{host}"
    elif getattr(space_info, "subdomain", None):
        direct_url = f"https://{space_info.subdomain}.hf.space"

    deployment_record = {
        "run_id": RUN_ID,
        "authenticated_account": identity.get("name"),
        "model_repo_id": MODEL_REPO_ID,
        "space_repo_id": SPACE_REPO_ID,
        "model_commit": str(model_commit),
        "space_commit": str(space_commit),
        "space_stage": final_stage,
        "space_hardware_request": selected_hardware,
        "stage_history": stage_history,
        "hub_url": hub_url,
        "direct_url": direct_url,
    }
    atomic_write_json(deployment_record, DRIVE_RUN_DIR / "huggingface_deployment.json")
    atomic_write_json(deployment_record, PROJECT_DIR / "reports" / "huggingface_deployment.json")
    mark_stage("huggingface_deployed", **deployment_record)

    display(HTML(
        f'<a href="{hub_url}" target="_blank" style="display:inline-block;padding:13px 20px;'
        'background:#f97316;color:white;font-weight:800;border-radius:11px;text-decoration:none;margin:8px 8px 8px 0;">'
        'Open the DefectSpotter Hugging Face Space</a>'
        + (f'<a href="{direct_url}" target="_blank" style="display:inline-block;padding:13px 20px;'
           'background:#0f172a;color:white;font-weight:800;border-radius:11px;text-decoration:none;margin:8px 0;">'
           'Open the direct app</a>' if direct_url else '')
    ))
    print(json.dumps(deployment_record, indent=2))


## 22 · Launch the complete Colab preview

The permanent deployment is the Hugging Face Space above. This final cell also launches the same rich interface inside Colab for debugging. It includes interactive EDA and performance charts, data provenance, upload/webcam/clipboard input, streaming camera inspection, numerical anomaly maps, heat-map overlays, batch analysis, threshold economics, drift views, and named API endpoints.


In [ ]:
from __future__ import annotations

import importlib
import os
import sys
import time
from IPython.display import HTML, display

required_ui_files = [
    ARTIFACT_DIR / "onnx" / "mobilenet_v3_small_patch.onnx",
    REPORT_DIR / "metrics_by_category_method.csv",
    REPORT_DIR / "operating_metrics.csv",
    REPORT_DIR / "latency.csv",
    REPORT_DIR / "artifact_sizes.csv",
    REPORT_DIR / "drift_stress_test.csv",
    REPORT_DIR / "image_statistics.csv",
    REPORT_DIR / "dataset_composition.csv",
]
missing_ui_files = [str(path) for path in required_ui_files if not path.exists()]
if missing_ui_files:
    raise FileNotFoundError("The rich UI requires completed artifacts. Missing:\n- " + "\n- ".join(missing_ui_files))

os.environ["DEFECTSPOTTER_ARTIFACTS"] = str(ARTIFACT_DIR)
os.environ["DEFECTSPOTTER_PROJECT_ROOT"] = str(PROJECT_DIR)
os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import gradio_app
importlib.invalidate_caches()
importlib.reload(gradio_app)

if "demo" in globals():
    try:
        demo.close()
    except Exception:
        pass

demo = gradio_app.build_app(ARTIFACT_DIR, project_root=PROJECT_DIR)
print("Launching the complete DefectSpotter preview")
print("API endpoints: /inspect_part, /inspect_batch, /health")

if AUTO_LAUNCH_GRADIO:
    launch_error = None
    launch_result = None
    for attempt in range(1, 3):
        try:
            launch_result = gradio_app.launch_app(
                demo,
                share=GRADIO_SHARE,
                server_port=None,
                prevent_thread_lock=True,
                inline=True,
            )
            break
        except Exception as exc:
            launch_error = exc
            print(f"Gradio launch attempt {attempt} failed: {exc}")
            if attempt == 1:
                time.sleep(3)
    if launch_result is None:
        raise RuntimeError("Gradio could not launch. Rerun only this preview cell; no training is required.") from launch_error

    local_url = getattr(demo, "local_url", None)
    public_url = getattr(demo, "share_url", None)
    if isinstance(launch_result, (tuple, list)) and len(launch_result) >= 3:
        _, returned_local, returned_public = launch_result[:3]
        local_url = returned_local or local_url
        public_url = returned_public or public_url
    print(f"Local URL: {local_url}")
    print(f"Temporary public URL: {public_url}")
    target_url = public_url or local_url
    if target_url:
        display(HTML(
            f'<a href="{target_url}" target="_blank" style="display:inline-block;padding:13px 20px;'
            'background:#f97316;color:white;font-weight:800;border-radius:11px;text-decoration:none;margin:8px 0;">'
            'Open the Colab DefectSpotter preview</a>'
        ))
else:
    print("AUTO_LAUNCH_GRADIO is False. Set it to True and rerun only this cell.")


## Completion checklist

A successful run now produces all of the following:

- validated MVTec AD data for bottle, screw, and carpet;
- EDA tables, interactive-ready evidence, montages, acquisition statistics, and defect-mask analysis;
- synthetic scratches, stains and cut-paste anomalies for label-free calibration;
- PatchCore-style, aligned-mean, convolutional-VAE and ensemble artifacts;
- image AUROC, pixel AUROC, AUPRO, operating rates, threshold-gap and cost evidence;
- ONNX verification, latency, artifact-footprint and drift reports;
- a local ZIP and persistent Drive run folder;
- attributed research examples for immediate UI demonstration;
- `bhkr0901/defectspotter-artifacts` as the model/report backend;
- `bhkr0901/defectspotter` as the permanent Hugging Face frontend and inference API;
- a rich Gradio experience with upload, clipboard, webcam snapshot and streaming webcam inspection, anomaly maps, heat-map localization, charts, EDA, batch summaries, drift monitoring and `/inspect_part`, `/inspect_batch`, `/health` endpoints.

The research dataset remains non-commercial, and a real production release still requires plant-owned data, camera qualification, line-specific validation and an approved human response for CHECK, drift and system failure.
